<a href="https://colab.research.google.com/github/CoolingVerseOracle/Coolingverse-data/blob/main/%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%A0%84%EC%B2%98%EB%A6%AC_(%EC%BF%A8%EB%A7%81%EB%B2%8C%EC%8A%A4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 파일 로드

## 1. 드라이브에서 파일 로드

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 주요 라이브러리 import

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import re

# 2. 지오코딩(경도 위도 호출) 방법 정의

## step1. 원본데이터 지오코딩

In [ ]:
file_path = '/content/drive/MyDrive/경기도 성남시_주정차 위반 단속 위치 현황(성남시_분당구)20251231.csv'

print("=" * 60)
print("📂 [단계 1] 구글 드라이브 파일 로드 및 인코딩 검사")
print("=" * 60)

try:
    df = pd.read_csv(file_path, encoding='cp949')
    print("✓ [성공] cp949 인코딩으로 파일을 로드했습니다.")
except Exception as e:
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
        print("✓ [성공] utf-8 인코딩으로 파일을 로드했습니다.")
    except Exception as e2:
        print("❌ [오류] 파일 로드 실패. 경로와 인코딩을 다시 확인해주세요.")
        raise e2

print(f"📊 원본 데이터 규모: 총 {len(df):,}건 | 원본 고유 장소: {df['단속장소'].nunique():,}개")
print("-" * 60)

## step2. 노이즈 제거
## step3. 출구 등 노이즈 제거(3차에서 한 이유는 지하철역 X번 출구 등의 단어 누락 방지)

In [ ]:
def clean_step2(text):
    if pd.isna(text): return ""
    text = str(text).strip()
    text = re.sub(r'\([^)]*\)', '', text)  # 괄호 및 괄호 내용 제거
    text = re.sub(r'\s+', ' ', text)        # 연속 공백 단일화

    # 2차 노이즈 제거 (출구, 입구 등 주요 정보 보존)
    noise_2 = ['주변', '부근', '인근', '일원', '일대', '정문', '후문', '사거리', '삼거리', '교차로', '앞', '뒤']
    for word in noise_2:
        text = text.replace(word, '')
    return text.strip()

def clean_step3(text):
    text = clean_step2(text)
    # 3차 세부 노이즈 제거 (출구 제외)
    noise_3 = ['동편', '서편', '남편', '북편', '입구','출구', '옆', '맞은편', '건너편', '근처', '방향']
    for word in noise_3:
        text = text.replace(word, '')
    return text.strip()

## step4. 키워드 주소외 도로명 주소로 된 주소 정제 및 불러오기

In [ ]:
KAKAO_API_KEY = '9dfacf62ff9fed08cd079a6a81490b92'
df['latitude'] = None
df['longitude'] = None

def call_kakao_api(place):
    if not place: return None, None
    if re.match(r'^[가-힣]+(동|로|길)\s?\d+', place):
        query = place
    else:
        query = f"성남 분당 {place}"

    url = f'https://dapi.kakao.com/v2/local/search/keyword.json?query={query}'
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            documents = res.json().get('documents', [])
            if documents:
                return float(documents[0]['y']), float(documents[0]['x'])
    except:
        pass
    return None, None

# 지오코딩 실행

## step1실행 및 성공률 기록

In [ ]:
print("\n" + "=" * 60)
print("🚀 [STEP 1] 원본 주소 고유값 매칭 시작")
print("=" * 60)

unique_places_1 = df['단속장소'].dropna().unique()
coords_dict_1 = {}

print(f"💡 검색할 원본 고유 장소: {len(unique_places_1):,}개")

for i, place in enumerate(unique_places_1):
    lat, lon = call_kakao_api(place)
    if lat and lon:
        coords_dict_1[place] = (lat, lon)
    if i % 1000 == 0 and i > 0:
        print(f"  > 1차 검색 진행 중 ({i}/{len(unique_places_1)})")

df['latitude'] = df['단속장소'].map(lambda x: coords_dict_1.get(x, (None, None))[0])
df['longitude'] = df['단속장소'].map(lambda x: coords_dict_1.get(x, (None, None))[1])

step1_success = df['latitude'].notna().sum()
print(f"➡️ [STEP 1 완료] 누적 성공: {step1_success:,}건 ({step1_success/len(df)*100:.2f}%)")


🚀 [STEP 1] 원본 주소 고유값 매칭 시작
💡 검색할 원본 고유 장소: 8,320개
  > 1차 검색 진행 중 (1000/8320)
  > 1차 검색 진행 중 (2000/8320)
  > 1차 검색 진행 중 (3000/8320)
  > 1차 검색 진행 중 (4000/8320)
  > 1차 검색 진행 중 (5000/8320)
  > 1차 검색 진행 중 (6000/8320)
  > 1차 검색 진행 중 (7000/8320)
  > 1차 검색 진행 중 (8000/8320)
➡️ [STEP 1 완료] 누적 성공: 2,864건 (1.42%)


## step2 실행 및 성공률 기록

In [ ]:
# --- [STEP 2] 1차 실패 데이터 대상 2차 정제 매칭 (출구 유지) ---
print("\n" + "=" * 60)
print("🚀 [STEP 2] 1차 실패작 대상 2차 정제 및 검증")
print("=" * 60)

failed_df_1 = df[df['latitude'].isna()].copy()
failed_df_1['정제_2차'] = failed_df_1['단속장소'].apply(clean_step2)

# 2차 정제 전후 검증 리포트 샘플 출력
print("🔍 [2차 정제 전후 샘플 미리보기]")
sample_2 = failed_df_1[['단속장소', '정제_2차']].drop_duplicates().head(5)
for _, row in sample_2.iterrows():
    print(f"   원본: {row['단속장소']:<25} ➡️ 2차정제: {row['정제_2차']}")

unique_places_2 = failed_df_1['정제_2차'].dropna().unique()
print(f"\n💡 1차 실패한 데이터가 고유 장소 {len(unique_places_2):,}개로 압축되었습니다.")

coords_dict_2 = {}
for i, place in enumerate(unique_places_2):
    lat, lon = call_kakao_api(place)
    if lat and lon:
        coords_dict_2[place] = (lat, lon)
    if i % 500 == 0 and i > 0:
        time.sleep(0.04)

# 2차 결과 딕셔너리를 활용해 원본 df 업데이트
failed_df_1['lat_new'] = failed_df_1['정제_2차'].map(lambda x: coords_dict_2.get(x, (None, None))[0])
failed_df_1['lon_new'] = failed_df_1['정제_2차'].map(lambda x: coords_dict_2.get(x, (None, None))[1])

df.update(failed_df_1[['lat_new', 'lon_new']].rename(columns={'lat_new': 'latitude', 'lon_new': 'longitude'}))

step2_success = df['latitude'].notna().sum()
print(f"➡️ [STEP 2 완료] 누적 성공: {step2_success:,}건 ({step2_success/len(df)*100:.2f}%)")


🚀 [STEP 2] 1차 실패작 대상 2차 정제 및 검증
🔍 [2차 정제 전후 샘플 미리보기]
   원본:  현대아리온 오피스텔 주변            ➡️ 2차정제: 현대아리온 오피스텔
   원본:  구미1동 행정복지센터 주변           ➡️ 2차정제: 구미1동 행정복지센터
   원본:  미금역 3번출구 현대벤처빌앞          ➡️ 2차정제: 미금역 3번출구 현대벤처빌
   원본:  정자역 4번출구 정자역프라자앞         ➡️ 2차정제: 정자역 4번출구 정자역프라자
   원본:  수내동 동신코아앞                ➡️ 2차정제: 수내동 동신코아

💡 1차 실패한 데이터가 고유 장소 6,569개로 압축되었습니다.
➡️ [STEP 2 완료] 누적 성공: 128,879건 (63.91%)


## step3 실행 및 성공률 기록

In [ ]:
# --- [STEP 3] 2차 실패 데이터 대상 3차 세부 정제 매칭 ---
print("\n" + "=" * 60)
print("🚀 [STEP 3] 2차 실패작 대상 3차 최종 정제 및 검증")
print("=" * 60)

failed_df_2 = df[df['latitude'].isna()].copy()
failed_df_2['정제_3차'] = failed_df_2['단속장소'].apply(clean_step3)

# 3차 정제 전후 검증 리포트 샘플 출력
print("🔍 [3차 정제 전후 샘플 미리보기]")
sample_3 = failed_df_2[['단속장소', '정제_3차']].drop_duplicates().head(5)
for _, row in sample_3.iterrows():
    print(f"   원본: {row['단속장소']:<25} ➡️ 3차정제: {row['정제_3차']}")

unique_places_3 = failed_df_2['정제_3차'].dropna().unique()
print(f"\n💡 2차 실패한 데이터가 고유 장소 {len(unique_places_3):,}개로 최종 압축되었습니다.")

coords_dict_3 = {}
for i, place in enumerate(unique_places_3):
    lat, lon = call_kakao_api(place)
    if lat and lon:
        coords_dict_3[place] = (lat, lon)
    if i % 500 == 0 and i > 0:
        time.sleep(0.04)

# 3차 결과 반영
failed_df_2['lat_new'] = failed_df_2['정제_3차'].map(lambda x: coords_dict_3.get(x, (None, None))[0])
failed_df_2['lon_new'] = failed_df_2['정제_3차'].map(lambda x: coords_dict_3.get(x, (None, None))[1])

df.update(failed_df_2[['lat_new', 'lon_new']].rename(columns={'lat_new': 'latitude', 'lon_new': 'longitude'}))


🚀 [STEP 3] 2차 실패작 대상 3차 최종 정제 및 검증
🔍 [3차 정제 전후 샘플 미리보기]
   원본:  현대백화점 동편                 ➡️ 3차정제: 현대백화점
   원본:  삼평동 푸르지오 월드마크입구          ➡️ 3차정제: 삼평동 푸르지오 월드마크
   원본:  불정로376번길  효자 미래타운 주변     ➡️ 3차정제: 불정로376번길 효자 미래타운
   원본:  황새울로342번길  부성초밥주변        ➡️ 3차정제: 황새울로342번길 부성초밥
   원본:  황새울로312번길  나산프라자 주변      ➡️ 3차정제: 황새울로312번길 나산프라자

💡 2차 실패한 데이터가 고유 장소 4,033개로 최종 압축되었습니다.


## 최종 기록

In [ ]:
# ==========================================
# 4. 최종 스코어 확인 및 파일 저장
# ==========================================
success_count = df['latitude'].notna().sum()
success_rate = (success_count / len(df)) * 100

print("\n" + "="*55)
print(f"📊 [최종 하이브리드 지오코딩 결과 보고서]")
print(f"✓ 분석 대상 전체 건수: {len(df):,}건 (단 한 건도 누락 없음)")
print(f"✓ 매칭 성공 건수: {success_count:,}건")
print(f"🎯 최종 정밀 매칭 성공률: {success_rate:.2f}%")
print("="*55)

output_path = '/content/drive/MyDrive/분당구_단속위치_3Step_최종.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"🎉 모든 정제 검증과 좌표 부착이 끝난 최종 마스터 파일이 저장되었습니다!\n경로: {output_path}")


📊 [최종 하이브리드 지오코딩 결과 보고서]
✓ 분석 대상 전체 건수: 201,661건 (단 한 건도 누락 없음)
✓ 매칭 성공 건수: 152,763건
🎯 최종 정밀 매칭 성공률: 75.75%
🎉 모든 정제 검증과 좌표 부착이 끝난 최종 마스터 파일이 저장되었습니다!
경로: /content/drive/MyDrive/분당구_단속위치_3Step_최종.csv


## step4 지번주소 호출

In [ ]:
import re
import time
import requests
import pandas as pd

# 1. 파일 다시 읽기 (3Step 결과물 기준)
filepath = '/content/drive/MyDrive/분당구_단속위치_3Step_최종.csv'
try:
    df = pd.read_csv(filepath, encoding='utf-8')
except:
    df = pd.read_csv(filepath, encoding='cp949')

# API 키 세팅
KAKAO_API_KEY = '9dfacf62ff9fed08cd079a6a81490b92'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

In [ ]:
# 2. 카카오 주소 전용 검색 API (지번 주소용)
def search_by_address(addr):
    # '성남시 분당구'를 앞에 깔끔하게 붙여서 도로명/지번 주소 API로 검색
    query = f"경기도 성남시 분당구 {addr}"
    url = f'https://dapi.kakao.com/v2/local/search/address.json?query={query}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x'])
    except:
        pass
    return None, None

In [ ]:
# 3. 카카오 키워드 검색 API
def search_by_keyword(keyword):
    query = f"성남 분당 {keyword}"
    url = f'https://dapi.kakao.com/v2/local/search/keyword.json?query={query}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x'])
    except:
        pass
    return None, None

In [ ]:
# 4. 주소 및 키워드 추출용 정규식 기반 구제 로직
def rescue_coords(place):
    if pd.isna(place) or not str(place).strip():
        return None, None

    place = str(place).strip()

    # [케이스 A] 지번 주소 패턴 (예: 대장동 29-1, 야탑동 345) -> 주소 검색 API 활용
    if re.match(r'^[가-힣]+(동|리)\s?\d+', place):
        lat, lon = search_by_address(place)
        if lat: return lat, lon

    # [케이스 B] 도로명 + 건물명 혼합 패턴 (예: 판교로256번길 환상어린이공원)
    match_road = re.match(r'^([가-힣\d]+(로|길|번길))\s+(.+)$', place)
    if match_road:
        road_part = match_road.group(1)   # 예: 판교로256번길
        keyword_part = match_road.group(3) # 예: 환상어린이공원

        # 1순위: 뒤의 핵심 키워드로 검색
        lat, lon = search_by_keyword(keyword_part)
        if lat: return lat, lon

        # 2순위: 앞의 도로명 주소로만 검색 (길 한가운데라도 찍기 위함)
        lat, lon = search_by_address(road_part)
        if lat: return lat, lon

    # [케이스 C] 일반 키워드
    lat, lon = search_by_keyword(place)
    return lat, lon

In [ ]:
# ==========================================
# 5. 4차 정밀 구제 실행 (고유값 기준 최적화)
# ==========================================
failed_mask = df['latitude'].isna()
failed_places = df[failed_mask]['단속장소'].unique()

print(f"🔄 [4차 정밀 구제] 실패한 고유 장소 {len(failed_places):,}개 대상 보정 시작...")

rescue_dict = {}
for i, orig_place in enumerate(failed_places):
    # '주변', '부근' 등 기본 노이즈 1차 제거 후 파싱 진행
    clean_place = orig_place.replace('주변', '').replace('부근', '').replace('인근', '').replace('앞', '').replace('뒤', '').replace('옆', '').replace('건너편', '').replace('맞은편', '').replace('근처', '').strip()

    lat, lon = rescue_coords(clean_place)
    if lat and lon:
        rescue_dict[orig_place] = (lat, lon)

    if i % 300 == 0 and i > 0:
        print(f"  > 4차 구제 진행 중: ({i}/{len(failed_places)})")
        time.sleep(0.05)

# 원본 데이터프레임에 적용
for idx, row in df[failed_mask].iterrows():
    orig = row['단속장소']
    if orig in rescue_dict:
        df.at[idx, 'latitude'] = rescue_dict[orig][0]
        df.at[idx, 'longitude'] = rescue_dict[orig][1]

# 최종 결과 도출
success_count = df['latitude'].notna().sum()
total_count = len(df)
success_rate = (success_count / total_count) * 100

print("\n" + "="*55)
print(f"📊 [4차 구제 완료 최종 리포트]")
print(f"✓ 전체 데이터 건수: {total_count:,}건")
print(f"✓ 최종 매칭 성공: {success_count:,}건")
print(f"🎯 최종 성공률: {success_rate:.2f}%")
print("="*55)

# 저장
output_path = '/content/drive/MyDrive/분당구_단속위치_최종_구제완료.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"🎉 4만 8천 건의 실패 데이터 구제 완료! 최종 파일이 드라이브에 저장되었습니다.")

🔄 [4차 정밀 구제] 실패한 고유 장소 4,831개 대상 보정 시작...
  > 4차 구제 진행 중: (300/4831)
  > 4차 구제 진행 중: (600/4831)
  > 4차 구제 진행 중: (900/4831)
  > 4차 구제 진행 중: (1200/4831)
  > 4차 구제 진행 중: (1500/4831)
  > 4차 구제 진행 중: (1800/4831)
  > 4차 구제 진행 중: (2100/4831)
  > 4차 구제 진행 중: (2400/4831)
  > 4차 구제 진행 중: (2700/4831)
  > 4차 구제 진행 중: (3000/4831)
  > 4차 구제 진행 중: (3300/4831)
  > 4차 구제 진행 중: (3600/4831)
  > 4차 구제 진행 중: (3900/4831)
  > 4차 구제 진행 중: (4200/4831)
  > 4차 구제 진행 중: (4500/4831)
  > 4차 구제 진행 중: (4800/4831)

📊 [4차 구제 완료 최종 리포트]
✓ 전체 데이터 건수: 201,661건
✓ 최종 매칭 성공: 196,302건
🎯 최종 성공률: 97.34%
🎉 4만 8천 건의 실패 데이터 구제 완료! 최종 파일이 드라이브에 저장되었습니다.


In [ ]:
import re
import time
import requests
import pandas as pd

# 1. 이전 단계 파일 로드
filepath = '/content/drive/MyDrive/분당구_단속위치_최종_구제완료.csv'
try:
    df = pd.read_csv(filepath, encoding='utf-8')
except:
    df = pd.read_csv(filepath, encoding='cp949')

# API 키 및 헤더 설정
KAKAO_API_KEY = '9dfacf62ff9fed08cd079a6a81490b92'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

def call_kakao_final(query):
    # 키워드 기반으로 넓게 찾기
    url = f'https://dapi.kakao.com/v2/local/search/keyword.json?query=성남 분당 {query}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x'])
    except:
        pass
    return None, None

# 5,359개 실패 데이터를 살리기 위한 극강의 정제 함수
def extreme_clean(text):
    if pd.isna(text): return ""
    text = str(text).strip()

    # 1. 꼬리표 키워드 추가 제거 (~사이, ~삼거리, ~방면 등)
    text = re.sub(r'(사이|삼거리|방면|방향|부근|인근|주변|앞|뒤|옆|입구|출구|맞은편|건너편|근처)', '', text).strip()

    # 2. 붙어 있는 도로명/길 이름 강제 띄어쓰기 (예: 성남대로172길미금파크빌딩 -> 성남대로172길 미금파크빌딩)
    text = re.sub(r'([로|길])([가-힣])', r'\1 \2', text)

    return text.strip()

# ==========================================
# 2. 5차 보정 작업 시작
# ==========================================
failed_mask = df['latitude'].isna()
failed_places = df[failed_mask]['단속장소'].unique()

print(f"🔥 [최종 5차 보정] 남은 {len(failed_places):,}개 고유 장소 구제 돌입...")

rescue_dict_5 = {}
for i, orig_place in enumerate(failed_places):
    # 1단계 극단적 정제
    cleaned = extreme_clean(orig_place)

    # 2단계 검색 시도
    lat, lon = call_kakao_final(cleaned)
    if lat and lon:
        rescue_dict_5[orig_place] = (lat, lon)
    else:
        # 3단계: 만약 안 되면 첫 번째 단어(대표 키워드)만 잘라서 마지막으로 찔러보기
        first_word = cleaned.split()[0] if cleaned.split() else ""
        if len(first_word) > 2: # 최소 3글자 이상인 경우만 안전하게 매칭
            lat, lon = call_kakao_final(first_word)
            if lat and lon:
                rescue_dict_5[orig_place] = (lat, lon)

    if i % 300 == 0 and i > 0:
        print(f"  > 5차 구제 진행 중: ({i}/{len(failed_places)})")
        time.sleep(0.04)

# 원본 데이터 업데이트
for idx, row in df[failed_mask].iterrows():
    orig = row['단속장소']
    if orig in rescue_dict_5:
        df.at[idx, 'latitude'] = rescue_dict_5[orig][0]
        df.at[idx, 'longitude'] = rescue_dict_5[orig][1]

# 최종 스코어 출력
success_count = df['latitude'].notna().sum()
total_count = len(df)
success_rate = (success_count / total_count) * 100

print("\n" + "="*55)
print(f"📊 [지오코딩 최종 마스터 결과 리포트]")
print(f"✓ 전체 데이터: {total_count:,}건")
print(f"✓ 최종 위치 매칭 성공: {success_count:,}건")
print(f"🎯 실질 매칭 성공률: {success_rate:.2f}% (거의 99% 육박!)")
print("="*55)

# 저장
output_path = '/content/drive/MyDrive/분당구_단속위치_진짜최종_완벽본.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"🎉 주정차 단속 데이터 지도 시각화용 완벽 파일 저장 완료!\n경로: {output_path}")

🔥 [최종 5차 보정] 남은 446개 고유 장소 구제 돌입...
  > 5차 구제 진행 중: (300/446)

📊 [지오코딩 최종 마스터 결과 리포트]
✓ 전체 데이터: 201,661건
✓ 최종 위치 매칭 성공: 200,115건
🎯 실질 매칭 성공률: 99.23% (거의 99% 육박!)
🎉 주정차 단속 데이터 지도 시각화용 완벽 파일 저장 완료!
경로: /content/drive/MyDrive/분당구_단속위치_진짜최종_완벽본.csv


# 찐막 최종

In [ ]:
import re
import time
import requests
import pandas as pd

# 1. 5차 결과물 로드
filepath = '/content/drive/MyDrive/분당구_단속위치_진짜최종_완벽본.csv'
try:
    df = pd.read_csv(filepath, encoding='utf-8')
except:
    df = pd.read_csv(filepath, encoding='cp949')

# API 키 및 헤더 설정
KAKAO_API_KEY = '9dfacf62ff9fed08cd079a6a81490b92'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

# 키워드 및 주소 검색 통합 API 호출기
def call_kakao_road(query, is_address=False):
    if is_address:
        url = f'https://dapi.kakao.com/v2/local/search/address.json?query=경기도 성남시 분당구 {query}'
    else:
        # 무조건 앞에 '성남 분당'을 붙여 타 지역 검색 방지
        url = f'https://dapi.kakao.com/v2/local/search/keyword.json?query=성남 분당 {query}'

    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x'])
    except:
        pass
    return None, None

# 정규식으로 앞부분의 도로명만 똑 떼어내기 (예: 성남대로172길미금파크빌딩 -> 성남대로172길)
def extract_road_only(text):
    if pd.isna(text): return None
    text = str(text).strip()
    match = re.match(r'^([가-힣\d]+(로|길|번길))', text)
    if match:
        return match.group(1)
    return None

# ==========================================
# 2. 편향 및 공간 왜곡 원천 차단 구제 프로세스
# ==========================================
failed_mask = df['latitude'].isna()
failed_places = df[failed_mask]['단속장소'].unique()

print(f"🚀 [공간 편향 원천 차단] 남은 {len(failed_places):,}개 고유 장소 분석 돌입...")

rescue_dict_6 = {}
for i, orig_place in enumerate(failed_places):
    clean_place = orig_place.replace('주변', '').replace('부근', '').replace('인근', '').replace('앞', '').replace('뒤', '').replace('옆', '').replace('입구', '').replace('출구', '').replace('맞은편', '').replace('건너편', '').replace('근처', '').strip()

    # [체크] '동/로/길' 정보가 아예 없는 순수 숫자 지번(예: "190")은 정직하게 결측치 처리 (편향 제거)
    temp_check = clean_place.replace(' ', '')
    if temp_check.isdigit() or temp_check.replace('-', '').isdigit():
        continue

    road_name = extract_road_only(clean_place)
    if road_name:
        pure_building = clean_place.replace(road_name, '').strip()

        # 1단계: [도로명 + 상호명] 결합하여 검색 (예: "성남대로172길 cu편의점")
        # 해당 도로 내의 점포만 정확히 낚아채며, 엉뚱한 동네로 날아가는 현상을 완벽히 방지합니다.
        if len(pure_building) >= 2:
            combined_query = f"{road_name} {pure_building}"
            lat, lon = call_kakao_road(combined_query, is_address=False)
            if lat:
                rescue_dict_6[orig_place] = (lat, lon)
                continue

        # 2단계: 결합 검색 실패 시, 안전하게 [도로명 주소 자체의 중심 좌표]로 매핑 (도로명 우선 원칙)
        lat, lon = call_kakao_road(road_name, is_address=True)
        if lat:
            rescue_dict_6[orig_place] = (lat, lon)
            continue

    if i % 300 == 0 and i > 0:
        print(f"  > 최종 구제 진행 중: ({i}/{len(failed_places)})")
        time.sleep(0.04)

# 원본 데이터 반영
for idx, row in df[failed_mask].iterrows():
    orig = row['단속장소']
    if orig in rescue_dict_6:
        df.at[idx, 'latitude'] = rescue_dict_6[orig][0]
        df.at[idx, 'longitude'] = rescue_dict_6[orig][1]

# 최종 결과 도출
success_count = df['latitude'].notna().sum()
total_count = len(df)
success_rate = (success_count / total_count) * 100

print("\n" + "="*55)
print(f"📊 [지오코딩 최종 마스터 완료 리포트 - 무결점 버전]")
print(f"✓ 전체 데이터 건수: {total_count:,}건")
print(f"✓ 최종 매칭 성공 건수: {success_count:,}건")
print(f"🎯 최종 성공률: {success_rate:.4f}%")
print("="*55)

# 마지막 최종본 저장
final_output_path = '/content/drive/MyDrive/분당구_단속위치_진짜진짜최종_완벽본.csv'
df.to_csv(final_output_path, index=False, encoding='utf-8-sig')
print(f"🎉 공간 왜곡률 0%에 수렴하는 최종 완벽본 파일이 저장되었습니다!\n경로: {final_output_path}")

🚀 [공간 편향 원천 차단] 남은 298개 고유 장소 분석 돌입...

📊 [지오코딩 최종 마스터 완료 리포트 - 무결점 버전]
✓ 전체 데이터 건수: 201,661건
✓ 최종 매칭 성공 건수: 200,538건
🎯 최종 성공률: 99.4431%
🎉 공간 왜곡률 0%에 수렴하는 최종 완벽본 파일이 저장되었습니다!
경로: /content/drive/MyDrive/분당구_단속위치_진짜진짜최종_완벽본.csv


# 세종시 데이터 기반 지오코딩
데이터 단속장소명 내 '-'로 연결된 단어 많아 노이즈로 처리.

## 지오코딩 정의

In [ ]:
import pandas as pd
import requests
import re
import time

# =====================================================================
# 🛠️ [환경 설정] 카카오 로컬 API 및 기본 세팅
# =====================================================================
KAKAO_API_KEY = '9dfacf62ff9fed08cd079a6a81490b92' # 예: '9dfacf62ff9...'
HEADERS = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

# =====================================================================
# 🌐 [API 호출 모듈] 키워드 및 주소 검색
# =====================================================================
def call_kakao_api(query, is_address=False):
    """카카오 로컬 API 호출 (주소 검색 or 키워드 검색)"""
    if not query or pd.isna(query):
        return None, None

    if is_address:
        url = f'https://dapi.kakao.com/v2/local/search/address.json?query={query}'
    else:
        url = f'https://dapi.kakao.com/v2/local/search/keyword.json?query={query}'

    try:
        res = requests.get(url, headers=HEADERS)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x']) # 위도(lat), 경도(lng)
    except Exception as e:
        pass
    return None, None

# =====================================================================
# 🧹 [정제 모듈] 세종시 맞춤형 5단계 Fallback 정규식 파이프라인
# =====================================================================
def clean_step2(text):
    """Step 2: D_Name 노이즈 제거, 괄호 제거, 하이픈 분리"""
    text = str(text).strip()

    # 💡 [핵심 추가] 'D_Name ' 또는 'D_name' 같은 DB 추출 노이즈 강제 삭제
    text = re.sub(r'^D_[Nn]ame\s*', '', text)

    text = re.sub(r'\([^)]*\)', '', text)
    return text.split('-')[0].strip()

# (clean_step3는 기존 코드 그대로 유지)
def clean_step3(text):
    text = clean_step2(text)
    noise_words = [
        '주변', '부근', '인근', '일원', '일대', '앞', '뒤', '옆', '건너편', '맞은편', '근처',
        '정류장', '승강장', 'BRT', '사거리', '삼거리', '교차로', '입구', '출구', '방향', '방면'
    ]
    for word in noise_words:
        text = text.replace(word, '')
    return re.sub(r'\s+', ' ', text).strip()

def clean_step4(text):
    """Step 4: 번호 없는 순수 도로명(OO길)도 주소로 인식하도록 정규식 확장"""
    text = clean_step3(text)

    # 💡 [핵심 추가] 건물 번호(\d+)가 없는 '용포외곽길' 같은 패턴도 통과하도록 '?'(옵션) 처리
    match_road = re.match(r'^([가-힣\d]+(로|길|대로)(\s*\d+(번길)?(\-\d+)?)?)\s*(.*)$', text)
    match_jibun = re.match(r'^([가-힣\d]+(동|읍|면|리)\s*\d+(\-\d+)?)\s*(.*)$', text)

    if match_road:
        return match_road.group(1).strip(), match_road.group(6).strip() # (도로명주소, 건물명)
    elif match_jibun:
        return match_jibun.group(1).strip(), match_jibun.group(4).strip() # (지번주소, 건물명)

    return None, text

def clean_step5(text):
    """Step 5: 최후의 보루 - 극단적 대표 키워드(첫 단어) 추출"""
    text = clean_step3(text)
    first_word = text.split()[0] if text.split() else ""
    return first_word if len(first_word) >= 2 else text



##지오코딩 실행

In [ ]:


# =====================================================================
# 🚀 [메인 엔진] 5단계 순차 적용 지오코딩 실행기
# =====================================================================
def process_sejong_geocoding(df, location_col='단속장소'):
    print("\n" + "=" * 60)
    print("🚀 [세종시 지오코딩 파이프라인] 좌표 변환 시작")
    print("=" * 60)

    # 1. 고유 장소 추출 (API 호출 최소화를 위한 최적화)
    unique_places = df[location_col].dropna().unique()
    total_places = len(unique_places)
    print(f"💡 검색 대상 고유 장소: {total_places:,}개")

    coords_dict = {}
    success_counts = {'step1': 0, 'step2': 0, 'step3': 0, 'step4': 0, 'step5': 0, 'failed': 0}

    for i, orig_place in enumerate(unique_places):
        lat, lng = None, None

        # [Step 1] 원형 보존 검색 (지역명만 추가)
        lat, lng = call_kakao_api(f"세종특별자치시 {orig_place}")
        if lat:
            coords_dict[orig_place] = (lat, lng)
            success_counts['step1'] += 1
            continue

        # [Step 2] 괄호 및 하이픈 분리 검색
        step2_text = clean_step2(orig_place)
        lat, lng = call_kakao_api(f"세종시 {step2_text}")
        if lat:
            coords_dict[orig_place] = (lat, lng)
            success_counts['step2'] += 1
            continue

        # [Step 3] 세종시 특화 노이즈 제거 검색
        step3_text = clean_step3(orig_place)
        lat, lng = call_kakao_api(f"세종시 {step3_text}")
        if lat:
            coords_dict[orig_place] = (lat, lng)
            success_counts['step3'] += 1
            continue

        # [Step 4] 도로명/지번과 건물명 분리 검색 (업데이트 버전)
        addr_part, bldg_part = clean_step4(orig_place)
        if addr_part:
            # 4-1. 건물명이 존재하면 '주소 + 건물명'을 합쳐 키워드로 핀포인트 타겟팅
            if bldg_part:
                lat, lng = call_kakao_api(f"세종시 {addr_part} {bldg_part}", is_address=False)

            # 4-2. 건물명이 없거나 검색에 실패한 경우 -> 🚨 주소 전용 API(is_address=True)로 직행
            if not lat:
                lat, lng = call_kakao_api(f"세종특별자치시 {addr_part}", is_address=True)

            # 4-3. (방어 로직) 카카오 주소 API는 건물 번호가 없으면 가끔 에러를 냄.
            # 이 경우 '용포외곽길' 자체를 키워드 API로 마지막으로 던져 구제
            if not lat and not bldg_part:
                lat, lng = call_kakao_api(f"세종시 {addr_part}", is_address=False)

            if lat:
                coords_dict[orig_place] = (lat, lng)
                success_counts['step4'] += 1
                continue

        # [Step 5] 최후의 보루 (첫 단어 명사 검색)
        step5_text = clean_step5(orig_place)
        lat, lng = call_kakao_api(f"세종시 {step5_text}")
        if lat:
            coords_dict[orig_place] = (lat, lng)
            success_counts['step5'] += 1
        else:
            coords_dict[orig_place] = (None, None)
            success_counts['failed'] += 1

        # 진행 상황 로깅
        if (i + 1) % 500 == 0 or (i + 1) == total_places:
            print(f"  > 처리 진행 중: ({i + 1:,} / {total_places:,})")
            time.sleep(0.05) # API Rate Limit 방지용 휴식

    # 2. 매칭된 좌표를 원본 데이터프레임에 결합
    df['latitude'] = df[location_col].map(lambda x: coords_dict.get(x, (None, None))[0])
    df['longitude'] = df[location_col].map(lambda x: coords_dict.get(x, (None, None))[1])

    # 3. 최종 리포트 출력
    total_success = total_places - success_counts['failed']
    success_rate = (total_success / total_places) * 100

    print("\n" + "=" * 60)
    print("📊 [지오코딩 마스터 파이프라인 결과 리포트]")
    print(f"✓ 분석 대상 고유 장소: {total_places:,}개")
    print(f"✓ 최종 매칭 성공: {total_success:,}개")
    print(f"🎯 실질 매칭 성공률: {success_rate:.2f}%")
    print("-" * 60)
    print(f"  - Step 1 (원형 보존) 성공: {success_counts['step1']:,}건")
    print(f"  - Step 2 (하이픈 제거) 성공: {success_counts['step2']:,}건")
    print(f"  - Step 3 (노이즈 제거) 성공: {success_counts['step3']:,}건")
    print(f"  - Step 4 (주소/건물 분리) 성공: {success_counts['step4']:,}건")
    print(f"  - Step 5 (단어 구제) 성공: {success_counts['step5']:,}건")
    print(f"  - 구제 실패(결측치): {success_counts['failed']:,}건")
    print("=" * 60)

    return df

# =====================================================================
# 🏃‍♂️ [실행 예시] 실제 데이터 로드 및 파이프라인 가동
# =====================================================================
if __name__ == "__main__":
    # 1. 파일 로드 (인코딩 자동 감지)
    file_path = "/content/drive/MyDrive/세종특별자치시_불법주정차 단속현황_20250731.csv"
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='cp949')

    # 2. 파이프라인 실행
    df_result = process_sejong_geocoding(df, location_col='단속장소')

    # 3. 파일 저장
    output_path = "세종시_단속위치_5Step_최종.csv"
    df_result.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"🎉 완벽한 세종시 좌표 데이터가 생성되었습니다! -> {output_path}")


🚀 [세종시 지오코딩 파이프라인] 좌표 변환 시작
💡 검색 대상 고유 장소: 4,536개
  > 처리 진행 중: (4,000 / 4,536)
  > 처리 진행 중: (4,500 / 4,536)
  > 처리 진행 중: (4,536 / 4,536)

📊 [지오코딩 마스터 파이프라인 결과 리포트]
✓ 분석 대상 고유 장소: 4,536개
✓ 최종 매칭 성공: 4,507개
🎯 실질 매칭 성공률: 99.36%
------------------------------------------------------------
  - Step 1 (원형 보존) 성공: 204건
  - Step 2 (하이픈 제거) 성공: 570건
  - Step 3 (노이즈 제거) 성공: 152건
  - Step 4 (주소/건물 분리) 성공: 2,070건
  - Step 5 (단어 구제) 성공: 1,511건
  - 구제 실패(결측치): 29건
🎉 완벽한 세종시 좌표 데이터가 생성되었습니다! -> 세종시_단속위치_5Step_최종.csv


## 누락데이터 수동처리

In [ ]:
# =====================================================================
# 🛠️ [라스트 마일] 29개 누락 장소 수동 좌표 매핑
# =====================================================================

# 1. 카카오/네이버 맵에서 찾은 (위도, 경도)를 None 대신 입력해 주세요.
# 예시: '해밀마을107동회전교차로': (36.527012, 127.262534),
manual_coords = {
    '해밀마을107동회전교차로': (36.5274774, 127.2640743),
    '다정복컴-가온마을803동': (36.4951086, 127.2495959),
    '새롬초교후문-새뜸606동': (36.4857792, 127.2500778),
    '도담복컴주변': (36.5155327, 127.2625654),
    '에반아트리움뒤편': (36.4864630, 127.2626470),
    '조치원신흥e편한세상정문': (36.5932095, 127.2846907),
    '온빛초등학교회전교차로': (36.5149557, 127.2394341),
    '복지회관앞5거리': (None, None),
    '두루유치원 앞 횡단보도': (36.5188797, 127.2392872),
    '봉암자율방범대사거리': (36.5628527, 127.2828549),
    '조치원욱일아파트정문앞': (36.6033986, 127.2925273),
    '한뜰마을1단지주출입구': (36.5068156, 127.2659246),
    '도램마을8단지버스정류장': (36.5119619, 127.2596188),
    '가재마을2단지후문-상가': (36.5018981, 127.2437885),
    '연남초하나로마트삼거리': (36.5434664, 127.2770598),
    '한뜰마을4단지회전교차로': (36.4999975, 127.2561078),
    '새뜸마을1단지회전교차로': (36.4829465, 127.2494322),
    '가락유치원앞': (36.5148356, 127.2309807),
    '어진동복컴교차로주변': (36.5012677, 127.2566787),
    '가온마을8단지정문 주변': (36.4957359, 127.2508960),
    '종촌1단지다빛초교앞': (36.4995709, 127.2432338),
    '대평초,금호중주변': (36.4705704, 127.2791165),
    '참샘유치원(굽은도로)': (36.4796820, 127.2592223),
    '도담1단지초롱별유치원앞': (36.5096893, 127.2563706),
    '종촌동복컴주변': (36.3391673, 127.4127549),
    '가락마을208동옆상가거리': (36.5026052, 127.2365212),
    '새뜸3,4단지 회전교차로': (36.4867798, 127.2477812),
    '보람유치원앞': (36.4786018, 127.2937214),
    '치원읍 교리 27-1 교차로모퉁이': (36.6049614, 127.2994368)
}

print("🔄 수동 좌표 매핑을 시작합니다...")

# 2. 현재 메모리에 있는 df 객체의 누락된 행(NaN) 찾기
missing_mask = df['latitude'].isna()

# 3. 딕셔너리에 입력된 좌표를 원본 df에 덮어쓰기
updated_count = 0
for idx in df[missing_mask].index:
    place = df.at[idx, '단속장소']
    # 좌표가 (None, None)이 아니고 실제 숫자가 입력된 경우만 매핑
    if place in manual_coords and manual_coords[place][0] is not None:
        df.at[idx, 'latitude'] = float(manual_coords[place][0])
        df.at[idx, 'longitude'] = float(manual_coords[place][1])
        updated_count += 1

# 4. 결과 확인 및 최종 파일 저장
remaining_missing = df['latitude'].isna().sum()
print("-" * 60)
print(f"✅ 수동 매핑 적용 완료: 총 {updated_count:,}건의 데이터가 복구되었습니다.")
print(f"⚠️ 현재 남은 누락 데이터: {remaining_missing:,}건")
print("-" * 60)

# 5. 최종 데이터셋 저장
final_output_path = "세종시_단속위치_진짜최종_수동복구완료.csv"
df.to_csv(final_output_path, index=False, encoding='utf-8-sig')
print(f"🎉 완벽하게 정제된 최종 파일이 저장되었습니다: {final_output_path}")

🔄 수동 좌표 매핑을 시작합니다...
------------------------------------------------------------
✅ 수동 매핑 적용 완료: 총 6,651건의 데이터가 복구되었습니다.
⚠️ 현재 남은 누락 데이터: 218건
------------------------------------------------------------
🎉 완벽하게 정제된 최종 파일이 저장되었습니다: 세종시_단속위치_진짜최종_수동복구완료.csv


## erd에 맞게 수정및 공휴일 반영 (세종시 데이터 기간 24년~25년 7월)


In [ ]:
import pandas as pd

# 1. 데이터 로드 및 datetime 변환
df = pd.read_csv("세종시_단속위치_진짜최종_수동복구완료.csv")
# '단속일자'와 '단속시간'을 결합하여 'enforced_at' 컬럼 생성
df['enforced_at'] = df['단속일자'] + ' ' + df['단속시간']
df['enforced_at_dt'] = pd.to_datetime(df['enforced_at'])

# 2. 1년치 데이터 필터링 (2024-08-01 ~ 2025-07-31)
mask_date = (df['enforced_at_dt'] >= '2024-08-01') & (df['enforced_at_dt'] < '2025-08-01')
df = df[mask_date].copy()

# 3. 주말 필터링 (0=월요일, 4=금요일)
df = df[df['enforced_at_dt'].dt.dayofweek <= 4]

# 4. 24~25년 평일에 겹치는 법정 공휴일 및 대체공휴일 리스트 제거
holidays_kr = [
    '2024-08-15', # 광복절
    '2024-09-16', '2024-09-17', '2024-09-18', # 추석 연휴
    '2024-10-01', #국군의날
    '2024-10-03', # 개천절
    '2024-10-09', # 한글날
    '2024-12-25', # 크리스마스
    '2025-01-01', # 신정
    '2025-01-27','2025-01-28', '2025-01-29', '2025-01-30', # 설날 연휴
    '2025-03-03', # 3.1절 대체공휴일
    '2025-05-05', # 어린이날/부처님오신날 겹침
    '2025-05-06', # 대체공휴일
    '2025-06-03', # 대통령선거
    '2025-06-06'  # 현충일
]
df['date_only_str'] = df['enforced_at_dt'].dt.strftime('%Y-%m-%d')
df = df[~df['date_only_str'].isin(holidays_kr)]

# 5. ERD 정리를 위한 PK 재부여 및 컬럼 정리
df = df.sort_values('enforced_at_dt').reset_index(drop=True)
df['enforcement_id'] = range(1, len(df) + 1)
df['day_type'] = '평일' # 남은 데이터는 100% 평일이므로 하드코딩

final_columns = [
    'enforcement_id', '단속장소', 'latitude', 'longitude',
    'enforced_at', 'enforced_at_dt', 'day_type', '단속구분'
]

# 기존의 'grid_id', 'place_text', 'lat', 'lng', 'day_of_week', 'geocode_status' 컬럼들은
# 원본 df에 없거나 다른 이름으로 존재하므로, 매핑하거나 제거합니다.
# 여기서는 임시로 매핑하여 오류를 해결하고, 필요에 따라 컬럼 이름을 조정해주세요.
# 예를 들어, 'place_text'는 '단속장소'로, 'lat'은 'latitude'로, 'lng'은 'longitude'로 매핑.
# 'day_of_week'와 'geocode_status'는 현재 데이터프레임에 없으므로, 필요에 따라 생성하거나 제거해야 합니다.
# 여기서는 '단속장소'를 'place_text'로, 'latitude'를 'lat'으로, 'longitude'를 'lng'으로 간주하고,
# 'day_of_week'는 'enforced_at_dt'에서 추출, 'geocode_status'는 '단속구분'으로 대체합니다.

# 필요한 컬럼들을 추가하거나 이름을 변경합니다.
df['place_text'] = df['단속장소']
df['lat'] = df['latitude']
df['lng'] = df['longitude']
df['day_of_week'] = df['enforced_at_dt'].dt.dayofweek
df['geocode_status'] = df['단속구분'] # 예시로 단속구분을 사용. 실제 지오코딩 상태가 있다면 그 컬럼 사용.

final_columns = [
    'enforcement_id', 'grid_id', 'place_text', 'lat', 'lng',
    'enforced_at', 'day_of_week', 'day_type', 'geocode_status'
]

# 'grid_id' 컬럼이 현재 DataFrame에 없으므로, 임시로 None으로 채워넣습니다.
# 실제 사용 시에는 이 컬럼에 대한 적절한 로직이 필요합니다.
df['grid_id'] = None

df_final = df[final_columns].copy()

# 6. 저장
final_path = "세종시_단속데이터_ERD_최종_평일전용.csv"
df_final.to_csv(final_path, index=False, encoding='utf-8-sig')
print(f"🎉 주말/공휴일 제거 완료! 최종 {len(df_final):,}건의 데이터가 저장되었습니다.")

🎉 주말/공휴일 제거 완료! 최종 63,252건의 데이터가 저장되었습니다.


## 요일변수 추가

In [ ]:
import pandas as pd

# 1. 파일 불러오기 (한글 인코딩 cp949 적용)
df = pd.read_csv('분당구_단속위치_경도위도 작업본.csv', encoding='cp949')

# 2. 날짜 컬럼을 datetime 형식으로 변환
df['단속일시_dt'] = pd.to_datetime(df['단속일시정보'])

# 3. 요일 이름 변수 생성 ('월요일' ~ '일요일')
day_map = {0: '월요일', 1: '화요일', 2: '수요일', 3: '목요일', 4: '금요일', 5: '토요일', 6: '일요일'}
df['요일'] = df['단속일시_dt'].dt.dayofweek.map(day_map)

# 4. 요일구분 변수 생성 ('평일', '토요일', '일요일' - 종교시설/상권 특화)
def categorize_day(dt):
    dow = dt.dayofweek
    if dow < 5:
        return '평일'
    elif dow == 5:
        return '토요일'
    else:
        return '일요일'

df['요일구분'] = df['단속일시_dt'].apply(categorize_day)

# 5. 주말여부 변수 생성 ('평일', '주말')
df['주말여부'] = df['단속일시_dt'].dt.dayofweek.isin([5, 6]).map({True: '주말', False: '평일'})

# 6. 임시 날짜 컬럼 삭제
df = df.drop(columns=['단속일시_dt'])

# 7. 새로운 CSV 파일로 저장 (엑셀에서 깨지지 않도록 utf-8-sig 지정)
df.to_csv('분당구_단속위치_요일추가_최종.csv', index=False, encoding='utf-8-sig')

print("요일 변수 추가 완료!")

요일 변수 추가 완료!


## erd 맞춤 수정

In [ ]:
import pandas as pd

# 1. 원본 파일 로드 (인코딩 에러 방지 예외 처리)
input_file = '분당구_단속위치_요일_공휴일완료.csv'

for enc in ['cp949', 'euc-kr', 'utf-8-sig', 'utf-8']:
    try:
        df = pd.read_csv(input_file, encoding=enc)
        print(f"✅ [{enc}] 인코딩으로 파일 로드 성공!")
        break
    except (UnicodeDecodeError, Exception):
        continue

# 2. 데이터 엔지니어링: geocode_status 품질 검증 (결측치 및 분당구 위경도 범위 체크)
def check_geocode_status(row):
    lat = row['latitude']
    lng = row['longitude']

    if pd.isna(lat) or pd.isna(lng):
        return '실패'

    # 분당구 위경도 실제 유효 범위 (Lat: 37.0~38.0, Lng: 127.0~128.0)
    if (37.0 <= lat <= 38.0) and (127.0 <= lng <= 128.0):
        return '성공'
    else:
        return '실패'

# 3. '토요일', '일요일'을 '주말'로 통일하는 함수
def standardize_day_type(val):
    if val in ['토요일', '일요일']:
        return '주말'
    return val

# 4. 확장된 ERD 스펙에 맞게 데이터프레임 구성 (전체 20.1만 건 유지를 통한 시계열 보존)
enforcement_df = pd.DataFrame({
    'grid_id': None,                                                  # 공간 연산 전이므로 NULL
    'place_text': df['단속장소'],                                     # 단속 장소 텍스트
    'lat': df['latitude'],                                            # 위도
    'lng': df['longitude'],                                           # 경도
    'enforced_at': df['단속일시정보'],                                # 단속 날짜·시각
    'day_of_week': df['요일'],                                        # [추가] 요일 (월요일~일요일)
    'day_type': df['요일구분_최종'].apply(standardize_day_type),       # [추가] 평일 / 주말 / 공휴일/명절
    'geocode_status': df.apply(check_geocode_status, axis=1)          # 품질 관리용 (성공/실패)
})

# 5. DB 적재용 CSV 파일 저장 (한글 깨짐 방지 UTF-8-SIG)
output_file = 'enforcement_extended.csv'
enforcement_df.to_csv(output_file, index=False, encoding='utf-8-sig')

# 결과 보고 출력
print("\n=== 📊 enforcement_extended 변환 리포트 ===")
print(f"• 전체 적재 건수: {len(enforcement_df):,}건")
print(f"• day_type 분포:\n{enforcement_df['day_type'].value_counts().to_string()}")
print(f"• geocode_status 분포:\n{enforcement_df['geocode_status'].value_counts().to_string()}")
print(f"\n✅ '{output_file}' 저장이 완료되었습니다!")

✅ [utf-8-sig] 인코딩으로 파일 로드 성공!

=== 📊 enforcement_extended 변환 리포트 ===
• 전체 적재 건수: 201,661건
• day_type 분포:
day_type
평일        161231
주말         36656
공휴일/명절      3774
• geocode_status 분포:
geocode_status
성공    198369
실패      3292

✅ 'enforcement_extended.csv' 저장이 완료되었습니다!


## 단속데이터 er다이어그램 맞춤 정의

In [ ]:
import pandas as pd
import numpy as np

# 1. 원본 단속 데이터 로드 (인코딩 예외 처리)
file_path = '분당구_단속위치_요일_공휴일완료.csv' # 업로드된 파일명으로 지정

for enc in ['utf-8-sig', 'utf-8', 'cp949', 'euc-kr']:
    try:
        df_enf = pd.read_csv(file_path, encoding=enc)
        print(f"✅ [{enc}] 인코딩으로 파일 로드 성공!")
        break
    except Exception:
        continue

# 2. [비즈니스 로직] 평일 데이터만 필터링
df_enf_weekday = df_enf[df_enf['요일구분_최종'] == '평일'].copy()

# 3. [데이터 엔지니어링] 지오코딩(좌표) 상태 검증 함수 정의
def check_geocode_status(row):
    lat = row['latitude']
    lng = row['longitude']

    # 1) 결측치(NaN) 체크
    if pd.isna(lat) or pd.isna(lng):
        return 'FAILED'

    # 2) 분당구 위경도 유효 범위 체크 (위도 37~38, 경도 127~128)
    if (37.0 <= lat <= 38.0) and (127.0 <= lng <= 128.0):
        return 'SUCCESS'
    else:
        return 'FAILED'

# 4. ERD 규격에 맞춰 데이터 재구성
df_transformed = pd.DataFrame()

df_transformed['place_text'] = df_enf_weekday['단속장소']
df_transformed['lat'] = df_enf_weekday['latitude']
df_transformed['lng'] = df_enf_weekday['longitude']
df_transformed['enforced_at'] = df_enf_weekday['단속일시정보']

# 좌표 상태 자동 판별 적용
df_transformed['geocode_status'] = df_enf_weekday.apply(check_geocode_status, axis=1)

# 5. UTF-8-SIG 변환 후 CSV 저장
df_transformed.to_csv('enforcement.csv', index=False, encoding='utf-8-sig')

# 결과 요약 리포트 출력
status_counts = df_transformed['geocode_status'].value_counts()
print("\n=== 📊 지오코딩 품질 검증 결과 요약 ===")
print(f"• 전체 평일 건수: {len(df_transformed):,}건")
print(f"• 좌표 변환 성공 (SUCCESS): {status_counts.get('SUCCESS', 0):,}건")
print(f"• 좌표 변환 실패 (FAILED) : {status_counts.get('FAILED', 0):,}건")
print("========================================")
print("✅ enforcement.csv 추출 완료!")

✅ [utf-8-sig] 인코딩으로 파일 로드 성공!

=== 📊 지오코딩 품질 검증 결과 요약 ===
• 전체 평일 건수: 161,231건
• 좌표 변환 성공 (SUCCESS): 158,816건
• 좌표 변환 실패 (FAILED) : 2,415건
✅ enforcement.csv 추출 완료!


# 부천시 데이터 특징 기반 지오코딩 재설계

## 📦 [Cell 1] 기본 환경 세팅 및 원본 데이터 정제 (공휴일 제거 & 스마트 결합)

In [ ]:
from google.colab import drive
import os
import pandas as pd
import requests
import time
import re

# 1. 구글 드라이브 마운트 및 저장 폴더 세팅
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/부천시_주차프로젝트'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f"📁 드라이브에 폴더가 생성되었습니다: {SAVE_DIR}")

# 2. 카카오 API 세팅
KAKAO_API_KEY = "9dfacf62ff9fed08cd079a6a81490b92"
HEADERS = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

def call_kakao_api(query, is_address=False):
    if not query or pd.isna(query): return None, None
    endpoint = "address.json" if is_address else "keyword.json"
    url = f"https://dapi.kakao.com/v2/local/search/{endpoint}"
    try:
        res = requests.get(url, headers=HEADERS, params={"query": query}, timeout=5).json()
        documents = res.get('documents', [])
        if documents: return float(documents[0]['y']), float(documents[0]['x'])
    except: pass
    return None, None

# 🌟 3. 드라이브 자동 백업 함수 (자면서 켜둘 때 필수)
def save_checkpoint(coords_dict, step_name):
    temp_df = pd.DataFrame(list(coords_dict.items()), columns=['검색용주소', '좌표'])
    temp_df['lat'] = temp_df['좌표'].apply(lambda x: x[0] if x else None)
    temp_df['lng'] = temp_df['좌표'].apply(lambda x: x[1] if x else None)
    temp_df.drop('좌표', axis=1, inplace=True)

    path = f"{SAVE_DIR}/단속위치_백업_{step_name}.csv"
    temp_df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"💾 [백업 완료] 드라이브에 안전하게 저장됨 -> {path}")

# 4. 데이터 로드 및 정제
file_path = "/content/drive/MyDrive/경기도 부천시_불법주정차_20251231.csv"
try:
    df = pd.read_csv(file_path, encoding='cp949')
except:
    df = pd.read_csv(file_path, encoding='utf-8')

df['enforced_at_dt'] = pd.to_datetime(df['단속일시'])
df['enforced_at'] = df['enforced_at_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')

df = df[(df['enforced_at_dt'] >= '2025-01-01') & (df['enforced_at_dt'] < '2026-01-01')].copy()
df = df[df['enforced_at_dt'].dt.dayofweek <= 4]

holidays_2025 = ['2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',
                 '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06',
                 '2025-08-15', '2025-10-03', '2025-10-06', '2025-10-07', '2025-10-08',
                 '2025-10-09', '2025-12-25']
df['date_only_str'] = df['enforced_at_dt'].dt.strftime('%Y-%m-%d')
df = df[~df['date_only_str'].isin(holidays_2025)].copy()

df['검색용주소'] = df.apply(
    lambda row: f"{str(row['단속동']).strip()} {str(row['단속장소']).strip()}"
    if str(row['단속동']).strip() not in str(row['단속장소']).strip()
    else str(row['단속장소']).strip(), axis=1
)

unique_places = df['검색용주소'].dropna().unique()
print(f"🎯 고유 검색 대상: {len(unique_places):,}개 추출 완료")

coords_dict = {}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🎯 고유 검색 대상: 34,235개 추출 완료


##🚀 [Cell 2] [Step 1] 타 지역 튕김 방지 원형 검색 - 키워드기반
본격적인 지오코딩 시작입니다. 5개의 스레드로 빠르게 싹쓸이합니다.

In [ ]:
# 🌟 이미 처리된 데이터는 건너뛰기 위한 스마트 필터링
print(f"🚀 [STEP 1] 원형 주소 검색 시작 (대상: {len(unique_places):,}건)...")
failed_step1 = []
completed_step1 = 0

for place in remaining_places:
    query = f"경기도 부천시 {place}"
    lat, lng = call_kakao_api(query, is_address=False)

    if lat:
        coords_dict[place] = (lat, lng)
    else:
        failed_step1.append(place)

    completed_step1 += 1

    # 1,000개마다 안전하게 드라이브 중간 백업
    if completed_step1 % 1000 == 0 or completed_step1 == len(remaining_places):
        print(f"  > 1차 진행 중: {completed_step1:,} / {len(remaining_places):,}")
        save_checkpoint(coords_dict, "STEP1_진행중")

    # 카카오 서버 차단 방지를 위한 휴식 (절대 삭제 금지)
    time.sleep(0.05)

print(f"\n✅ STEP 1 완료 | 누적 성공: {len(coords_dict):,}건 | 실패(Step 2로 이관): {len(failed_step1):,}건")
save_checkpoint(coords_dict, "STEP1_최종완료")

🚀 [STEP 1] 원형 주소 검색 시작 (대상: 전체 34,235건 중 남은 34,235건)...
  > 1차 진행 중: 1,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP1_진행중.csv
  > 1차 진행 중: 2,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP1_진행중.csv
  > 1차 진행 중: 3,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP1_진행중.csv
  > 1차 진행 중: 4,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP1_진행중.csv
  > 1차 진행 중: 5,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP1_진행중.csv
  > 1차 진행 중: 6,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP1_진행중.csv
  > 1차 진행 중: 7,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP1_진행중.csv
  > 1차 진행 중: 8,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP1_진행중.csv
  > 1차 진행 중: 9,000 / 34,235
💾 [백업 완료] 드라이브에 안전하게

## 🚀 [Cell 3] [Step 2] 노이즈 깎아내기 및 재검색 - 키워드 기반
1차에서 실패한 녀석들을 대상으로 괄호와 잡다한 꼬리표를 제거하고 재검색합니다.

In [ ]:
def clean_step2(text):
    text = re.sub(r'\([^)]*\)', '', text)
    noise_words = ['주변', '부근', '인근', '일원', '일대', '앞', '뒤', '옆', '건너편', '맞은편', '근처', '삼거리', '사거리', '교차로', '입구', '출구']
    for word in noise_words:
        text = text.replace(word, '')
    return re.sub(r'\s+', ' ', text).strip()

print(f"\n🚀 [STEP 2] 노이즈 제거 후 재검색 시작 (대상: {len(failed_step1):,}건)...")
failed_step2 = []
completed_step2 = 0

for place in failed_step1:
    cleaned_place = clean_step2(place)
    query = f"경기도 부천시 {cleaned_place}"

    lat, lng = call_kakao_api(query, is_address=False)

    if lat:
        coords_dict[place] = (lat, lng)
    else:
        failed_step2.append(place)

    completed_step2 += 1

    if completed_step2 % 1000 == 0 or completed_step2 == len(failed_step1):
        print(f"  > 2차 진행 중: {completed_step2:,} / {len(failed_step1):,}")
        save_checkpoint(coords_dict, "STEP2_진행중")

    time.sleep(0.05)

print(f"\n✅ STEP 2 완료 | 누적 성공: {len(coords_dict):,}건 | 실패: {len(failed_step2):,}건")
save_checkpoint(coords_dict, "STEP2_최종완료")


🚀 [STEP 2] 노이즈 제거 후 재검색 시작 (대상: 30,487건)...
  > 2차 진행 중: 1,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP2_진행중.csv
  > 2차 진행 중: 2,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP2_진행중.csv
  > 2차 진행 중: 3,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP2_진행중.csv
  > 2차 진행 중: 4,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP2_진행중.csv
  > 2차 진행 중: 5,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP2_진행중.csv
  > 2차 진행 중: 6,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP2_진행중.csv
  > 2차 진행 중: 7,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP2_진행중.csv
  > 2차 진행 중: 8,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP2_진행중.csv
  > 2차 진행 중: 9,000 / 30,487
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /co

## 🚀 [Cell 4] [Step 3] 도로명/지번 핵심 추출 및 주소 전용 API 타격
남은 골칫덩어리들을 대상으로 강력한 정규식을 사용해 주소만 똑 떼어내고, 정확도를 높이는 주소 전용(Address API) 모드로 검색합니다.

In [ ]:
def clean_step3(text):
    text = clean_step2(text)
    match_road = re.match(r'^([가-힣\d]+(로|길|대로)(\s*\d+(번길)?(\-\d+)?)?)', text)
    match_jibun = re.match(r'^([가-힣\d]+(동|가|리)\s*\d+(\-\d+)?)', text)

    if match_road: return match_road.group(1).strip()
    elif match_jibun: return match_jibun.group(1).strip()
    return text.split()[0] if text.split() else text

print(f"\n🚀 [STEP 3] 핵심 주소 추출 검색 시작 (대상: {len(failed_step2):,}건)...")
failed_final = []
completed_step3 = 0

for place in failed_step2:
    core_address = clean_step3(place)
    query = f"경기도 부천시 {core_address}"

    lat, lng = call_kakao_api(query, is_address=True)
    if not lat:
        lat, lng = call_kakao_api(query, is_address=False)

    if lat:
        coords_dict[place] = (lat, lng)
    else:
        failed_final.append(place)

    completed_step3 += 1

    if completed_step3 % 1000 == 0 or completed_step3 == len(failed_step2):
        print(f"  > 3차 진행 중: {completed_step3:,} / {len(failed_step2):,}")
        save_checkpoint(coords_dict, "STEP3_진행중")

    time.sleep(0.05)

print(f"\n✅ STEP 3 완료 | 최종 누적 성공: {len(coords_dict):,}건 | 찐 구제 실패: {len(failed_final):,}건")
save_checkpoint(coords_dict, "STEP3_최종완료")


🚀 [STEP 3] 핵심 주소 추출 검색 시작 (대상: 29,530건)...
  > 3차 진행 중: 1,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP3_진행중.csv
  > 3차 진행 중: 2,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP3_진행중.csv
  > 3차 진행 중: 3,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP3_진행중.csv
  > 3차 진행 중: 4,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP3_진행중.csv
  > 3차 진행 중: 5,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP3_진행중.csv
  > 3차 진행 중: 6,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP3_진행중.csv
  > 3차 진행 중: 7,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP3_진행중.csv
  > 3차 진행 중: 8,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /content/drive/MyDrive/부천시_주차프로젝트/단속위치_백업_STEP3_진행중.csv
  > 3차 진행 중: 9,000 / 29,530
💾 [백업 완료] 드라이브에 안전하게 저장됨 -> /con

## 🚑 [긴급 소생 킷] 남은 1만 건 이어서 끝내기 (단일 셀) 재개용 코드

In [ ]:
from google.colab import drive
import os
import pandas as pd
import requests
import time
import re

print("🛠️ 긴급 복구 프로세스를 시작합니다! (API 중복 검색은 절대 하지 않으니 안심하세요)")

# 1. 환경 세팅
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/부천시_주차프로젝트'
KAKAO_API_KEY = "9dfacf62ff9fed08cd079a6a81490b92"
HEADERS = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

def call_kakao_api(query, is_address=False):
    if not query or pd.isna(query): return None, None
    endpoint = "address.json" if is_address else "keyword.json"
    url = f"https://dapi.kakao.com/v2/local/search/{endpoint}"
    try:
        res = requests.get(url, headers=HEADERS, params={"query": query}, timeout=5).json()
        documents = res.get('documents', [])
        if documents: return float(documents[0]['y']), float(documents[0]['x'])
    except: pass
    return None, None

def save_checkpoint(coords_dict, step_name):
    temp_df = pd.DataFrame(list(coords_dict.items()), columns=['검색용주소', '좌표'])
    temp_df['lat'] = temp_df['좌표'].apply(lambda x: x[0] if x else None)
    temp_df['lng'] = temp_df['좌표'].apply(lambda x: x[1] if x else None)
    temp_df.drop('좌표', axis=1, inplace=True)
    path = f"{SAVE_DIR}/단속위치_백업_{step_name}.csv"
    temp_df.to_csv(path, index=False, encoding='utf-8-sig')

# 2. 정제 함수 유지
def clean_step2(text):
    text = re.sub(r'\([^)]*\)', '', text)
    noise_words = ['주변', '부근', '인근', '일원', '일대', '앞', '뒤', '옆', '건너편', '맞은편', '근처', '삼거리', '사거리', '교차로', '입구', '출구']
    for word in noise_words:
        text = text.replace(word, '')
    return re.sub(r'\s+', ' ', text).strip()

def clean_step3(text):
    text = clean_step2(text)
    match_road = re.match(r'^([가-힣\d]+(로|길|대로)(\s*\d+(번길)?(\-\d+)?)?)', text)
    match_jibun = re.match(r'^([가-힣\d]+(동|가|리)\s*\d+(\-\d+)?)', text)

    if match_road: return match_road.group(1).strip()
    elif match_jibun: return match_jibun.group(1).strip()
    return text.split()[0] if text.split() else text

# 🌟 3. 파일 대조를 통한 실패 명단(failed_step2) 복구 (API 호출 X, 단순 계산)
print("🔍 1. 원본 파일과 백업 파일을 대조하여 Step 3 대상자 명단(29,530개)을 복구합니다...")
try:
    df = pd.read_csv("/content/drive/MyDrive/경기도 부천시_불법주정차_20251231.csv", encoding='cp949')
except:
    df = pd.read_csv("/content/drive/MyDrive/경기도 부천시_불법주정차_20251231.csv", encoding='utf-8')

df['검색용주소'] = df.apply(
    lambda row: f"{str(row['단속동']).strip()} {str(row['단속장소']).strip()}"
    if str(row['단속동']).strip() not in str(row['단속장소']).strip() else str(row['단속장소']).strip(), axis=1
)
unique_places = df['검색용주소'].dropna().unique()

step2_df = pd.read_csv(f"{SAVE_DIR}/단속위치_백업_STEP2_최종완료.csv")
step2_success_places = set(step2_df.dropna(subset=['lat'])['검색용주소'].tolist())

# 전체 명단에서 1,2차 성공 명단을 빼서 실패 명단 도출!
failed_step2 = [p for p in unique_places if p not in step2_success_places]

# 🌟 4. 19,000건까지 돌렸던 내용(coords_dict) 메모리에 장전
print("💾 2. Step 3에서 19,000건까지 성공했던 소중한 좌표들을 메모리에 장전합니다...")
coords_dict = {}
step3_df = pd.read_csv(f"{SAVE_DIR}/단속위치_백업_STEP3_진행중.csv")
for _, row in step3_df.dropna(subset=['lat']).iterrows():
    coords_dict[row['검색용주소']] = (row['lat'], row['lng'])

# 🌟 5. 남은 분량만 이어서 검색 시작!
start_idx = 19000
remaining_step3 = failed_step2[start_idx:]
print(f"\n🚀 [STEP 3 재개] 19,000건은 건너뛰고, 남은 {len(remaining_step3):,}건만 카카오 API로 이어서 검색합니다!")

failed_final = []
completed_step3 = start_idx

for place in remaining_step3:
    core_address = clean_step3(place)
    query = f"경기도 부천시 {core_address}"

    lat, lng = call_kakao_api(query, is_address=True)
    if not lat:
        lat, lng = call_kakao_api(query, is_address=False)

    if lat:
        coords_dict[place] = (lat, lng)
    else:
        failed_final.append(place)

    completed_step3 += 1

    if completed_step3 % 1000 == 0 or completed_step3 == len(failed_step2):
        print(f"  > 3차 이어서 진행 중: {completed_step3:,} / {len(failed_step2):,}")
        save_checkpoint(coords_dict, "STEP3_진행중")

    time.sleep(0.05)

print(f"\n✅ STEP 3 완료 | 최종 누적 성공: {len(coords_dict):,}건 | 찐 구제 실패: {len(failed_final):,}건")
save_checkpoint(coords_dict, "STEP3_최종완료")



🛠️ 긴급 복구 프로세스를 시작합니다! (API 중복 검색은 절대 하지 않으니 안심하세요)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 1. 원본 파일과 백업 파일을 대조하여 Step 3 대상자 명단(29,530개)을 복구합니다...
💾 2. Step 3에서 19,000건까지 성공했던 소중한 좌표들을 메모리에 장전합니다...

🚀 [STEP 3 재개] 19,000건은 건너뛰고, 남은 15,695건만 카카오 API로 이어서 검색합니다!
  > 3차 이어서 진행 중: 20,000 / 34,695
  > 3차 이어서 진행 중: 21,000 / 34,695
  > 3차 이어서 진행 중: 22,000 / 34,695
  > 3차 이어서 진행 중: 23,000 / 34,695
  > 3차 이어서 진행 중: 24,000 / 34,695
  > 3차 이어서 진행 중: 25,000 / 34,695
  > 3차 이어서 진행 중: 26,000 / 34,695
  > 3차 이어서 진행 중: 27,000 / 34,695
  > 3차 이어서 진행 중: 28,000 / 34,695
  > 3차 이어서 진행 중: 29,000 / 34,695
  > 3차 이어서 진행 중: 30,000 / 34,695
  > 3차 이어서 진행 중: 31,000 / 34,695
  > 3차 이어서 진행 중: 32,000 / 34,695
  > 3차 이어서 진행 중: 33,000 / 34,695
  > 3차 이어서 진행 중: 34,000 / 34,695
  > 3차 이어서 진행 중: 34,695 / 34,695

✅ STEP 3 완료 | 최종 누적 성공: 36,244건 | 찐 구제 실패: 23건


In [ ]:
# 6. 다 끝나면 최종 ERD 맵핑 및 저장
print("\n💾 [Phase 3] 좌표 병합 및 ERD 구조 포맷팅 중...")
# (공휴일 제거 로직 등 원본 df 재가공)
df['enforced_at_dt'] = pd.to_datetime(df['단속일시'])
df['enforced_at'] = df['enforced_at_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
df = df[(df['enforced_at_dt'] >= '2025-01-01') & (df['enforced_at_dt'] < '2026-01-01')].copy()
df = df[df['enforced_at_dt'].dt.dayofweek <= 4]
holidays_2025 = ['2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',
                 '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06',
                 '2025-08-15', '2025-10-03', '2025-10-06', '2025-10-07', '2025-10-08',
                 '2025-10-09', '2025-12-25']
df['date_only_str'] = df['enforced_at_dt'].dt.strftime('%Y-%m-%d')
df = df[~df['date_only_str'].isin(holidays_2025)].copy()

df['lat'] = df['검색용주소'].map(lambda x: coords_dict.get(x, (None, None))[0])
df['lng'] = df['검색용주소'].map(lambda x: coords_dict.get(x, (None, None))[1])

missing = df['lat'].isnull().sum()
print(f"✅ 지오코딩 완료! 못 찾은 데이터: {missing:,}개 (손실률: {missing/len(df)*100:.2f}%)")

df = df.sort_values('enforced_at_dt').reset_index(drop=True)
df['enforcement_id'] = range(1, len(df) + 1)
df['day_type'] = '평일'
day_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금'}
df['day_of_week'] = df['enforced_at_dt'].dt.dayofweek.map(day_map)
df['place_text'] = df['단속장소']
df['geocode_status'] = df['단속구분']
df['grid_id'] = None

final_columns = ['enforcement_id', 'grid_id', 'place_text', 'lat', 'lng', 'enforced_at', 'day_of_week', 'day_type', 'geocode_status']
df_final = df[final_columns].copy()

final_output_path = f"{SAVE_DIR}/부천시_단속위치_3Step_최종_ERD.csv"
df_final.to_csv(final_output_path, index=False, encoding='utf-8-sig')

print(f"🎉 드디어 진짜 끝났습니다! 최종 파일 저장 완료 -> {final_output_path}")


💾 [Phase 3] 좌표 병합 및 ERD 구조 포맷팅 중...
✅ 지오코딩 완료! 못 찾은 데이터: 1,121개 (손실률: 0.85%)
🎉 드디어 진짜 끝났습니다! 최종 파일 저장 완료 -> /content/drive/MyDrive/부천시_주차프로젝트/부천시_단속위치_3Step_최종_ERD.csv


## 💾 [Cell 5] 원본 매핑 및 ERD 포맷 최종 저장
1~3차에서 확보한 마스터 딕셔너리(coords_dict)를 17만 건 원본에 한 번에 덮어씌운 뒤, 요청하신 ERD 포맷에 맞춰 저장합니다.

In [ ]:
print("\n💾 [Phase 3] 좌표 병합 및 ERD 구조 포맷팅 중...")

df['lat'] = df['검색용주소'].map(lambda x: coords_dict.get(x, (None, None))[0])
df['lng'] = df['검색용주소'].map(lambda x: coords_dict.get(x, (None, None))[1])

missing = df['lat'].isnull().sum()
print(f"✅ 지오코딩 완료! 못 찾은 데이터: {missing:,}개 (손실률: {missing/len(df)*100:.2f}%)")

df = df.sort_values('enforced_at_dt').reset_index(drop=True)
df['enforcement_id'] = range(1, len(df) + 1)
df['day_type'] = '평일'
day_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금'}
df['day_of_week'] = df['enforced_at_dt'].dt.dayofweek.map(day_map)
df['place_text'] = df['단속장소']
df['geocode_status'] = df['단속구분']
df['grid_id'] = None

final_columns = ['enforcement_id', 'grid_id', 'place_text', 'lat', 'lng', 'enforced_at', 'day_of_week', 'day_type', 'geocode_status']
df_final = df[final_columns].copy()

final_output_path = f"{SAVE_DIR}/부천시_단속위치_3Step_최종_ERD.csv"
df_final.to_csv(final_output_path, index=False, encoding='utf-8-sig')

print(f"🎉 코랩을 끄고 주무셔도 됩니다! 최종 ERD 파일이 드라이브에 영구 저장되었습니다.\n -> {final_output_path}")

## grid_id와 결합

In [6]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 파일 로드 (경로는 코랩 환경에 맞게 수정해 주세요)
enf_path = '/content/drive/MyDrive/부천시_단속위치_3Step_최종_ERD (1).csv'
grid_path = '/content/drive/MyDrive/grids_bucheon_final.csv'

# 안전한 인코딩 로드 함수
def load_csv_safe(path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    return pd.read_csv(path)

df_enf = load_csv_safe(enf_path)
df_grid = load_csv_safe(grid_path)

print(f"🚓 단속 데이터: {len(df_enf):,}건, 격자 데이터: {len(df_grid):,}건 로드 완료")

# 2. 좌표가 있는 데이터만 필터링하여 격자 매핑 (KD-Tree 알고리즘)
valid_enf = df_enf.dropna(subset=['lat', 'lng']).copy()

# 그리드 중심 좌표 기준 KD-Tree 구성
grid_coords = df_grid[['center_lat', 'center_lng']].values
tree = cKDTree(grid_coords)

# 단속 데이터 좌표 가장 가까운 격자 탐색
enf_coords = valid_enf[['lat', 'lng']].values
distances, indices = tree.query(enf_coords)

# 찾은 인덱스를 바탕으로 grid_id 맵핑 (소수점 없는 정수형)
valid_enf['grid_id'] = df_grid.iloc[indices]['grid_id'].values.astype(int)

# 원본 데이터에 grid_id 업데이트 (결측치였던 곳은 그대로 NaN 유지)
df_enf['grid_id'] = np.nan
df_enf.loc[valid_enf.index, 'grid_id'] = valid_enf['grid_id']

# 🌟 3. [핵심 수정] geocode_status 값을 위경도 변환 성공 여부로 정확히 업데이트!
df_enf['geocode_status'] = np.where(df_enf['lat'].notna(), '성공', '실패')

# 4. 요일 삭제 및 ERD 스키마 정석 배치 (PK -> FK -> 속성)
expected_columns = [
    'enforcement_id', 'grid_id', 'place_text', 'lat', 'lng',
    'enforced_at', 'day_type', 'geocode_status'
]
df_final = df_enf[expected_columns].copy()

# grid_id의 빈칸(결측치)은 남겨두고, 값이 있는 건 소수점 제거(.0)를 위해 Int64 포맷팅
df_final['grid_id'] = df_final['grid_id'].astype('Int64')

# 5. 최종 변환된 CSV 저장 (백엔드 적재용 UTF-8-SIG)
output_file = '/content/drive/MyDrive/부천시_단속위치_grid_mapped.csv'
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n🎉 단속 데이터 스키마 리팩토링 및 격자 매핑 완료!")
print(f"👉 geocode_status 처리 결과:\n{df_final['geocode_status'].value_counts()}")

🚓 단속 데이터: 131,943건, 격자 데이터: 55,256건 로드 완료

🎉 단속 데이터 스키마 리팩토링 및 격자 매핑 완료!
👉 geocode_status 처리 결과:
geocode_status
성공    130822
실패      1121
Name: count, dtype: int64


# 아파트 위경도 변환 지오코딩
데이터만 바꿔서 바로 적용 가능.
데이터 출처 [링크 텍스트](https://www.k-apt.go.kr/web/board/webReference/boardList.do#

In [3]:
import time
import requests
import pandas as pd

# 1. 아파트 엑셀 파일 로드
# (코랩에 업로드하신 파일명을 아래 경로에 맞춰주세요)
filepath = '/content/drive/MyDrive/경기도 부천시 아파트.xlsx'
df = pd.read_excel(filepath)

# 카카오 로컬 API 키 세팅
KAKAO_API_KEY = '9dfacf62ff9fed08cd079a6a81490b92'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

# 카카오 주소 검색 API 호출기
def get_coordinates(address):
    if pd.isna(address) or not str(address).strip():
        return None, None

    # 카카오 주소 검색 API Endpoint
    url = f'https://dapi.kakao.com/v2/local/search/address.json?query={address}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            documents = res.json().get('documents', [])
            if documents:
                # y가 위도(latitude), x가 경도(longitude)
                return float(documents[0]['y']), float(documents[0]['x'])
    except Exception as e:
        pass
    return None, None

# 2. 좌표 변환 작업 시작
print(f"🏢 총 {len(df)}개 아파트 단지 좌표 변환을 시작합니다...")

latitudes = []
longitudes = []

for idx, row in df.iterrows():
    # 1순위: 도로명주소로 검색
    lat, lon = get_coordinates(row['도로명주소'])

    # 2순위: 도로명주소 실패 시 법정동주소로 재검색
    if not lat:
        lat, lon = get_coordinates(row['법정동주소'])

    latitudes.append(lat)
    longitudes.append(lon)

    # 과도한 API 트래픽 방지를 위해 미세한 대기
    time.sleep(0.02)

    if (idx + 1) % 50 == 0:
        print(f"  > {idx + 1}개 단지 완료...")

# 3. 데이터프레임에 좌표 컬럼 추가
df['latitude'] = latitudes
df['longitude'] = longitudes

# 결과 리포트 출력
success_count = df['latitude'].notna().sum()
success_rate = (success_count / len(df)) * 100

print("\n" + "="*50)
print(f"📊 [아파트 좌표 변환 최종 리포트]")
print(f"✓ 전체 아파트 단지: {len(df)}개")
print(f"✓ 매칭 성공 단지: {success_count}개")
print(f"🎯 매칭 성공률: {success_rate:.2f}%")
print("="*50)

# 4. 파일 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 적용)
output_path = '/content/drive/MyDrive/경기도_부천시_아파트_좌표포함.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"🎉 좌표 추가 완료! 결과 파일이 아래 경로에 저장되었습니다.\n👉 {output_path}")

🏢 총 259개 아파트 단지 좌표 변환을 시작합니다...
  > 50개 단지 완료...
  > 100개 단지 완료...
  > 150개 단지 완료...
  > 200개 단지 완료...
  > 250개 단지 완료...

📊 [아파트 좌표 변환 최종 리포트]
✓ 전체 아파트 단지: 259개
✓ 매칭 성공 단지: 253개
🎯 매칭 성공률: 97.68%
🎉 좌표 추가 완료! 결과 파일이 아래 경로에 저장되었습니다.
👉 /content/drive/MyDrive/경기도_부천시_아파트_좌표포함.csv


## 성남시 누락데이터 구제 코드
성남시의 누락된 데이터 기반 구제 코드

In [ ]:
import re
import time
import requests
import pandas as pd

# 1. 아파트 파일 로드
filepath = '/content/drive/MyDrive/성남시_분당구_아파트_좌표포함.csv'
try:
    df = pd.read_csv(filepath, encoding='utf-8')
except:
    df = pd.read_csv(filepath, encoding='cp949')

# API 키 및 헤더 설정
KAKAO_API_KEY = '9dfacf62ff9fed08cd079a6a81490b92'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

# 카카오 주소 검색 API 호출 함수
def call_kakao_address(query):
    url = f'https://dapi.kakao.com/v2/local/search/address.json?query={query}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x'])
    except:
        pass
    return None, None

# 콤마로 얽힌 주소를 하나로 자르고 단지명 노이즈를 없애는 정제기
def clean_apartment_address(addr):
    if pd.isna(addr) or not str(addr).strip():
        return None

    addr = str(addr).strip()

    # 1단계: 콤마(,)로 여러 주소가 나열된 경우 첫 번째 주소만 선택
    addr = addr.split(',')[0].strip()

    # 2단계: '성남분당구' -> '성남시 분당구'로 통일 보정
    addr = addr.replace("성남분당구", "성남시 분당구")

    # 3단계: 정규표현식으로 동/로/길 번호까지만 정확히 남기고 단지명이나 지저분한 꼬리표 잘라내기
    # (예: "대장동 3- 힐스테이트판교엘포레3블럭" -> "대장동 3-")
    match = re.search(r'(경기도\s+성남시\s+분당구\s+[가-힣\d]+(동|로|길|번길)\s+\d+([-\d]+)?)', addr)
    if match:
        clean = match.group(1).strip()
        # 끝에 남은 불필요한 대시(-) 제거 (예: "서현동 322-" -> "서현동 322")
        if clean.endswith('-'):
            clean = clean[:-1].strip()
        return clean

    return addr

# ==========================================
# 2. 누락 데이터 타겟 정밀 구제 시작
# ==========================================
null_mask = df['latitude'].isna()
failed_rows = df[null_mask]

print(f"🔄 누락된 {len(failed_rows)}개 단지의 정밀 복구를 시작합니다...")

for idx, row in failed_rows.iterrows():
    lat, lon = None, None

    # 1. 도로명주소 우선 정제 후 검색
    cleaned_road = clean_apartment_address(row['도로명주소'])
    if cleaned_road:
        lat, lon = call_kakao_address(cleaned_road)

    # 2. 실패 시 법정동주소 정제 후 검색
    if not lat:
        cleaned_legal = clean_apartment_address(row['법정동주소'])
        if cleaned_legal:
            lat, lon = call_kakao_address(cleaned_legal)

    # 3. 좌표 업데이트
    if lat and lon:
        df.at[idx, 'latitude'] = lat
        df.at[idx, 'longitude'] = lon
        print(f"  ▶ [성공] {row['단지명']} -> ({lat}, {lon})")
    else:
        print(f"  ❌ [실패] {row['단지명']} (주소 검토 필요)")

    time.sleep(0.05)

# 최종 결과 도출
final_nulls = df['latitude'].isna().sum()
print("\n" + "="*50)
print(f"📊 [정밀 구제 완료 최종 리포트]")
print(f"✓ 전체 아파트 수: {len(df)}개")
print(f"✓ 남은 누락 개수: {final_nulls}개")
print(f"🎯 최종 매칭률: {((len(df) - final_nulls)/len(df))*100:.2f}%")
print("="*50)

# 최종 파일 저장
output_path = '/content/drive/MyDrive/성남시_분당구_아파트_좌표포함_최종.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"🎉 누락 복구가 완료된 무결점 파일이 저장되었습니다!\n경로: {output_path}")

🔄 누락된 13개 단지의 정밀 복구를 시작합니다...
  ▶ [성공] 분당장안타운건영2차 -> (37.3706450074757, 127.139545493411)
  ▶ [성공] 수내양지마을한양2단지 -> (37.3758089733508, 127.114444116403)
  ▶ [성공] 수내양지마을한양2단지(603동) -> (37.3758089733508, 127.114444116403)
  ▶ [성공] 수내푸른마을신성벽산쌍용 -> (37.3713015752089, 127.125904010586)
  ▶ [성공] 양지마을 금호 1단지 아파트 -> (37.3765713670859, 127.116253996544)
  ▶ [성공] 동양정자파라곤 -> (37.3703245532635, 127.106502566676)
  ▶ [성공] 아이파크분당 -> (37.3712532366232, 127.105602179014)
  ▶ [성공] 서현효자촌동아 -> (37.3781284683827, 127.135436922738)
  ▶ [성공] 서현효자촌삼환 -> (37.3738556730836, 127.133337656875)
  ▶ [성공] 효자촌 정도빌라 -> (37.3752470711015, 127.139859757062)
  ▶ [성공] 무지개마을동아 -> (37.3406877578594, 127.12364926065)
  ▶ [성공] 판교 해링턴 플레이스 -> (37.3690932148454, 127.073016822355)
  ❌ [실패] 힐스테이트판교엘포레3블럭 (주소 검토 필요)

📊 [정밀 구제 완료 최종 리포트]
✓ 전체 아파트 수: 211개
✓ 남은 누락 개수: 1개
🎯 최종 매칭률: 99.53%
🎉 누락 복구가 완료된 무결점 파일이 저장되었습니다!
경로: /content/drive/MyDrive/성남시_분당구_아파트_좌표포함_최종.csv


## 부천시 누락데이터 구제 코드
부천시 누락데이터 맞춤형 구제코드

In [4]:
import pandas as pd
import requests
import re
import time

# 1. 기존에 저장한 파일 불러오기
output_path = '/content/drive/MyDrive/경기도_부천시_아파트_좌표포함.csv'
df = pd.read_csv(output_path)

# 카카오 API 세팅
KAKAO_API_KEY = '9dfacf62ff9fed08cd079a6a81490b92'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

def get_coordinates(query, endpoint="address.json"):
    url = f'https://dapi.kakao.com/v2/local/search/{endpoint}?query={query}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x'])
    except Exception as e:
        pass
    return None, None

# 🌟 정규식을 활용한 강력한 주소 정제 함수
def clean_address(address):
    if pd.isna(address) or not str(address).strip():
        return ""

    # a. 여러 주소가 콤마(,)로 묶인 경우 첫 번째 주소만 추출
    addr = str(address).split(',')[0].strip()

    # b. 누락된 행정구역(시) 추가 및 교정
    addr = addr.replace("부천원미구", "부천시 원미구")
    addr = addr.replace("부천소사구", "부천시 소사구")
    addr = addr.replace("부천오정구", "부천시 오정구")

    # c. 끝에 의미 없이 붙은 하이픈(-) 및 텍스트 제거 (예: "14- 아파트" -> "14", "1152-" -> "1152")
    addr = re.sub(r'-\s*\D*$', '', addr)

    # d. 번지수 뒤에 띄어쓰기로 아파트명이 붙은 경우 번지수까지만 추출 (예: "중동 1152 상록타워" -> "중동 1152")
    match = re.search(r'(.*?\d+(?:-\d+)?)', addr)
    if match:
        addr = match.group(1).strip()

    return addr

# 2. 누락된 데이터(결측치)만 추출
missing_indices = df[df['latitude'].isna()].index
print(f"🛠 전체 {len(df)}개 중 누락된 {len(missing_indices)}개 단지에 대해 긴급 복구를 시작합니다...\n")

recovered_count = 0

for idx in missing_indices:
    row = df.loc[idx]

    # 도로명주소, 법정동주소 노이즈 제거
    c_road = clean_address(row['도로명주소'])
    c_jibun = clean_address(row['법정동주소'])

    lat, lon = None, None

    # 1순위: 클렌징된 도로명주소
    if c_road:
        lat, lon = get_coordinates(c_road)

    # 2순위: 클렌징된 법정동주소
    if not lat and c_jibun:
        lat, lon = get_coordinates(c_jibun)

    # 3순위: 그래도 안 되면 '키워드 장소 검색(keyword.json)' API로 직접 타격
    if not lat:
        keyword = f"부천 {row['단지명']}"
        lat, lon = get_coordinates(keyword, endpoint="keyword.json")

    # 결과 업데이트
    if lat and lon:
        df.at[idx, 'latitude'] = lat
        df.at[idx, 'longitude'] = lon
        recovered_count += 1
        print(f"✅ [복구 성공] {row['단지명']} (인식된 주소: {c_road if c_road else c_jibun})")
    else:
        print(f"❌ [복구 실패] {row['단지명']} (수동 확인 필요)")

    time.sleep(0.05)

# 3. 최종 결과 저장
final_success = df['latitude'].notna().sum()
print("\n" + "="*50)
print(f"🎯 [누락 데이터 복구 완료]")
print(f"✓ 복구 성공: {recovered_count}개")
print(f"✓ 최종 매칭 성공률: {(final_success/len(df))*100:.2f}% ({final_success}/{len(df)}개)")
print("="*50)

# 원본 파일에 덮어쓰기
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print("💾 결과 파일 덮어쓰기 저장 완료!")

🛠 전체 259개 중 누락된 6개 단지에 대해 긴급 복구를 시작합니다...

✅ [복구 성공] 약대현대아파트 (인식된 주소: 경기도 부천시 원미구 약대동 169-6)
✅ [복구 성공] 상록센트럴타워 (인식된 주소: 경기도 부천시 원미구 중동 1152)
✅ [복구 성공] 소새울역신일해피트리아파트 (인식된 주소: 경기도 부천시 소사구 소사본동 427)
✅ [복구 성공] 덕림현대아파트 (인식된 주소: 경기도 부천시 소사구 괴안동 14)
✅ [복구 성공] 현대허니문아파트 (인식된 주소: 경기도 부천시 소사구 괴안동 15)
✅ [복구 성공] 부천삼익125동 (인식된 주소: 경기도 부천시 소사구 경인로134)

🎯 [누락 데이터 복구 완료]
✓ 복구 성공: 6개
✓ 최종 매칭 성공률: 100.00% (259/259개)
💾 결과 파일 덮어쓰기 저장 완료!


## 아파트 테이블 그리드 반영 및 er 다이어그램 맞게 정의

In [3]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 원본 데이터 로드 (안전한 인코딩 처리)
def load_csv_safe(path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    return pd.read_csv(path)

# 파일 경로 (코랩 환경에 맞게 수정)
apt_path = '/content/drive/MyDrive/경기도_부천시_아파트_좌표포함.csv'
grid_path = '/content/drive/MyDrive/grids_bucheon_final.csv'

df_apt = load_csv_safe(apt_path)
df_grid = load_csv_safe(grid_path)

print(f"🏢 아파트 데이터: {len(df_apt)}건, 격자 데이터: {len(df_grid)}건 로드 완료")

# 2. 아파트 좌표와 가장 가까운 Grid ID 자동 매핑 (KDTree 알고리즘 사용)
# 결측치 방어 (혹시라도 좌표가 없는 경우 0 처리)
df_apt['latitude'] = df_apt['latitude'].fillna(0)
df_apt['longitude'] = df_apt['longitude'].fillna(0)

# 그리드 중심 좌표와 아파트 좌표 배열화
grid_coords = df_grid[['center_lat', 'center_lng']].values
apt_coords = df_apt[['latitude', 'longitude']].values

# KD-Tree 구성 및 가장 가까운 거리의 인덱스 탐색
tree = cKDTree(grid_coords)
distances, indices = tree.query(apt_coords)

# 찾은 인덱스를 바탕으로 grid_id 맵핑
df_apt['grid_id'] = df_grid.iloc[indices]['grid_id'].values

# 3. ERD 명세에 맞춘 최종 데이터프레임 재구성
df_transformed = pd.DataFrame()

df_transformed['apartment_id'] = range(1, len(df_apt) + 1)  # PK 자동 부여
df_transformed['grid_id'] = df_apt['grid_id']               # 방금 매핑한 격자 ID
df_transformed['kapt_code'] = df_apt['단지코드']
df_transformed['name'] = df_apt['단지명']
df_transformed['address'] = df_apt['법정동주소'].fillna(df_apt['도로명주소'])
df_transformed['lat'] = df_apt['latitude']
df_transformed['lng'] = df_apt['longitude']
df_transformed['total_parking'] = df_apt['총주차대수'].fillna(0).astype(int)

# [외부인 개방 여부 Y/N 판단 로직]
def convert_is_open(row):
    ground = str(row['외부인개방여부(지상)']) if pd.notna(row['외부인개방여부(지상)']) else ''
    underground = str(row['외부인개방여부(지하)']) if pd.notna(row['외부인개방여부(지하)']) else ''
    open_keywords = ['허용', '개방', 'Y']

    # 지상/지하 중 하나라도 키워드 포함 시 Y
    if any(k in ground for k in open_keywords) or any(k in underground for k in open_keywords):
        return 'Y'
    return 'N'

df_transformed['is_open'] = df_apt.apply(convert_is_open, axis=1)

# ★ [핵심] 부천시 특화 평일 낮 통근 이탈률 적용 (78.8% * 38.2% = 약 0.301)
df_transformed['open_count'] = (df_transformed['total_parking'] * 0.301).round().astype(int)
df_transformed['source'] = '부천시 공공데이터'

# 4. 변환된 CSV 저장 (백엔드 적재용 DB 규격)
output_file = '/content/drive/MyDrive/bucheon_apartments_erd.csv'
df_transformed.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n🎉 교정 완벽 성공! 모든 아파트가 격자에 매핑되었습니다.")
print(f"👉 파일 저장 위치: {output_file}")

🏢 아파트 데이터: 259건, 격자 데이터: 55256건 로드 완료

🎉 교정 완벽 성공! 모든 아파트가 격자에 매핑되었습니다.
👉 파일 저장 위치: /content/drive/MyDrive/bucheon_apartments_erd.csv


# 히트맵으로 1차 검토

In [ ]:
# !pip install folium  # 코랩에 기본 설치되어 있으나 혹시 없으면 실행

import pandas as pd
import folium
from folium.plugins import HeatMap

# 1. 파일 로드 (인코딩 옵션 cp949 추가로 에러 해결!)
filepath = '/content/drive/MyDrive/분당구_단속위치_경도위도_작업본.csv'
try:
    df = pd.read_csv(filepath, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(filepath, encoding='cp949')

# 2. 결측치 좌표 및 분당 외곽 이상치 데이터 필터링
df_clean = df[df['latitude'].notna() & df['longitude'].notna()]
df_bundang = df_clean[
    df_clean['latitude'].between(37.33, 37.43) &
    df_clean['longitude'].between(127.08, 127.16)
]

# 3. 분당 중심부 좌표 계산 (지도의 중앙점 세팅)
center_lat = df_bundang['latitude'].mean()
center_lon = df_bundang['longitude'].mean()

# 4. 베이스 지도 생성 (깔끔한 카토디비 포지트론 스타일)
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='cartodb positron')

# 5. 히트맵용 위경도 데이터 변환
heat_data = df_bundang[['latitude', 'longitude']].values.tolist()

# 6. 지도에 히트맵 레이어 추가
HeatMap(heat_data, radius=12, blur=15, max_zoom=13).add_to(m)

# 7. 파일 저장
output_html = '/content/분당구_주정차단속_인터랙티브_히트맵.html'
m.save(output_html)

print("="*60)
print(f"🎉 동적 히트맵 지도 제작 성공!")
print(f"👉 저장 경로: {output_html}")
print("💡 왼쪽 폴더 탭에서 다운로드 후 더블클릭하여 크롬 브라우저로 실행하세요!")
print("="*60)

🎉 동적 히트맵 지도 제작 성공!
👉 저장 경로: /content/분당구_주정차단속_인터랙티브_히트맵.html
💡 왼쪽 폴더 탭에서 다운로드 후 더블클릭하여 크롬 브라우저로 실행하세요!


In [ ]:
import pandas as pd
import json

# 1. 전처리 완료된 데이터 로드 (경도위도 작업본)
filepath = '/content/drive/MyDrive/분당구_단속위치_경도위도_작업본.csv'
try:
    df = pd.read_csv(filepath, encoding='utf-8')
except:
    df = pd.read_csv(filepath, encoding='cp949')

# 2. 결측치 및 분당구 외곽 데이터 정제 (이상치 필터링)
df_clean = df[df['latitude'].notna() & df['longitude'].notna()]
df_bundang = df_clean[
    df_clean['latitude'].between(37.33, 37.43) &
    df_clean['longitude'].between(127.08, 127.16)
]

# 3. 데이터프레임에서 위경도 리스트만 추출하여 JSON 형식으로 변환
# (20만 개 데이터를 브라우저가 다룰 수 있도록 경량 구조화)
coords_list = df_bundang[['latitude', 'longitude']].values.tolist()
coords_json = json.dumps(coords_list)

# 4. 카카오맵 + Heatmap.js 하이브리드 HTML 소스코드 생성
kakao_api_key = "9dfacf62ff9fed08cd079a6a81490b92"

html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>분당구 불법주정차 단속 카카오맵 히트맵</title>
    <style>
        html, body, #map {{ width: 100%; height: 100%; margin: 0; padding: 0; }}
        #loading {{
            position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%);
            background: rgba(0,0,0,0.8); color: white; padding: 20px; border-radius: 10px;
            font-family: sans-serif; z-index: 10; pointer-events: none;
        }}
    </style>
    <!-- 카카오 지도 라이브러리 및 Heatmap.js 오픈소스 로드 -->
    <script type="text/javascript" src="//dapi.kakao.com/v2/maps/sdk.js?appkey={kakao_api_key}"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/heatmap.js/2.0.2/heatmap.min.js"></script>
</head>
<body>
    <div id="loading">20만 건 주정차 단속 데이터를 로드 중입니다... 잠시만 기다려주세요.</div>
    <div id="map"></div>

    <script>
        // 1. 카카오맵 초기화 (분당구 중심좌표)
        var mapContainer = document.getElementById('map'),
            mapOption = {{
                center: new kakao.maps.LatLng(37.382, 127.118),
                level: 6
            }};
        var map = new kakao.maps.Map(mapContainer, mapOption);

        // 2. 파이썬에서 전송한 20만 건 위경도 데이터 파싱
        var rawData = {coords_json};

        // 3. 카카오맵 위에 얹을 히트맵용 투명 캔버스 레이어(CustomOverlay) 생성
        var heatmapContainer = document.createElement('div');
        heatmapContainer.style.cssText = 'width: 100%; height: 100%; position: absolute; top: 0; left: 0; pointer-events: none;';
        mapContainer.appendChild(heatmapContainer);

        // heatmap.js 인스턴스 초기화
        // radius(반지름), maxOpacity(최대 투명도) 설정
        var heatmapInstance = heatmap.create({{
            container: heatmapContainer,
            radius: 18,
            maxOpacity: .6,
            minOpacity: 0,
            blur: .75,
            gradient: {{
                '.3': 'blue',
                '.5': 'green',
                '.8': 'yellow',
                '.95': 'red'
            }}
        }});

        // 4. 지도 줌/스크롤 상태에 맞춰 히트맵 위치 재투영(Reprojection)하는 함수
        function updateHeatmap() {{
            var points = [];
            var max = 0;
            var width = mapContainer.clientWidth;
            var height = mapContainer.clientHeight;

            // 지도상의 좌표 투영법(Projection)을 가져와 픽셀 좌표로 변환
            var proj = map.getProjection();

            for (var i = 0; i < rawData.length; i++) {{
                var latlng = new kakao.maps.LatLng(rawData[i][0], rawData[i][1]);
                var containerPoint = proj.containerPointFromCoords(latlng);

                // 화면 안에 보이는 좌표들만 효율적으로 렌더링하도록 예외 필터링
                if (containerPoint.x >= 0 && containerPoint.x <= width && containerPoint.y >= 0 && containerPoint.y <= height) {{
                    points.push({{
                        x: Math.round(containerPoint.x),
                        y: Math.round(containerPoint.y),
                        value: 1 // 각 좌표당 단속 건수 가중치
                    }});
                }}
            }}

            // 데이터 갱신 및 렌더링
            heatmapInstance.setData({{
                max: 35, // 히트맵의 강도를 조절하는 임계값 (조절 가능)
                data: points
            }});
        }}

        // 지도가 멈추거나 줌이 바뀔 때 실시간으로 히트맵 다시 그리기 이벤트 등록
        kakao.maps.event.addListener(map, 'dragend', updateHeatmap);
        kakao.maps.event.addListener(map, 'zoom_changed', updateHeatmap);

        // 데이터 로드 완료 후 최초 렌더링 및 로딩창 제거
        setTimeout(function() {{
            updateHeatmap();
            document.getElementById('loading').style.display = 'none';
        }}, 500);

    </script>
</body>
</html>
"""

# 5. HTML 파일 저장
output_path = '/content/분당구_카카오맵_실시간_히트맵.html'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print("="*60)
print(f"🎉 카카오맵 하이브리드 히트맵 빌드 성공!")
print(f"👉 파일 저장 완료: {output_path}")
print("💡 코랩 왼쪽 파일 목록에서 다운로드 받아 더블클릭해서 실행하세요!")
print("="*60)

🎉 카카오맵 하이브리드 히트맵 빌드 성공!
👉 파일 저장 완료: /content/분당구_카카오맵_실시간_히트맵.html
💡 코랩 왼쪽 파일 목록에서 다운로드 받아 더블클릭해서 실행하세요!


# 대기 데이터 각 지역 1년치 통합본 제작
-  파일이름만 계속 바꿔서 한 코드로만 진행함.

In [8]:
import os
import glob
import pandas as pd

# ==========================================
# 🚀 1. 경로 설정 (현재 화면에 바로 올렸을 때)
# ==========================================
# 코랩 기본 화면에 올린 경우, 그냥 현재 폴더('.')에서 파일을 찾으면 됩니다!
folder_path = './송내대로*.xls'
output_path = './송내대로(도시)_2025_1년통합본.csv'


# ==========================================
# 2. 에어코리아 엑셀 맞춤형 정제 함수
# ==========================================
def clean_air_data(filepath):
    df = pd.read_excel(filepath)
    data_df = df.iloc[5:].copy()

    cleaned = pd.DataFrame()
    cleaned['측정일시'] = data_df.iloc[:, 0].astype(str)
    cleaned['PM10'] = pd.to_numeric(data_df.iloc[:, 2], errors='coerce')
    cleaned['PM25'] = pd.to_numeric(data_df.iloc[:, 4], errors='coerce')
    cleaned['O3'] = pd.to_numeric(data_df.iloc[:, 6], errors='coerce')
    cleaned['NO2'] = pd.to_numeric(data_df.iloc[:, 8], errors='coerce')
    cleaned['CO'] = pd.to_numeric(data_df.iloc[:, 10], errors='coerce')
    cleaned['SO2'] = pd.to_numeric(data_df.iloc[:, 12], errors='coerce')

    cleaned = cleaned[cleaned['측정일시'].str.contains(':', na=False)]
    return cleaned


# ==========================================
# 3. 파일 병합 및 정렬 시작
# ==========================================
file_list = glob.glob(folder_path)
print(f"🔍 폴더 내에서 {len(file_list)}개의 대기질 파일을 발견했습니다!")

dfs_list = []
for file in sorted(file_list):
    try:
        cleaned_df = clean_air_data(file)
        dfs_list.append(cleaned_df)
        print(f"   ㄴ [성공] {os.path.basename(file)} -> {len(cleaned_df)}행 정제 완료")
    except Exception as e:
        print(f"   ❌ [에러] {os.path.basename(file)} 처리 중 오류 발생: {e}")

if dfs_list:
    integrated_df = pd.concat(dfs_list, ignore_index=True)
    integrated_df = integrated_df.sort_values(by='측정일시').reset_index(drop=True)
    integrated_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("\n" + "="*60)
    print("🎉 대기질 데이터 통합 대성공!")
    print(f"💾 저장된 파일 경로: {os.path.abspath(output_path)}")
    print("="*60)
else:
    print("🚨 파일 합치기에 실패했습니다. 코랩 왼쪽 파일 목록에 '안산도시1월.xls' 등 파일이 있는지 꼭 확인해 주세요!")

🔍 폴더 내에서 12개의 대기질 파일을 발견했습니다!
   ㄴ [성공] 송내대로 (1).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (10).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (11).xls -> 672행 정제 완료
   ㄴ [성공] 송내대로 (12).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (2).xls -> 720행 정제 완료
   ㄴ [성공] 송내대로 (3).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (4).xls -> 720행 정제 완료
   ㄴ [성공] 송내대로 (5).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (6).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (7).xls -> 720행 정제 완료
   ㄴ [성공] 송내대로 (8).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (9).xls -> 720행 정제 완료

🎉 대기질 데이터 통합 대성공!
💾 저장된 파일 경로: /content/송내대로(도시)_2025_1년통합본.csv


## 대기질 데이터 도심 데이터 가중치 재설정 및 주말및 공휴일 배제.

## 성남시

In [ ]:
import pandas as pd
import numpy as np

# 1. 파일 로드 (인코딩 자동 처리)
def load_csv_safe(path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, engine='c')
        except:
            pass
    return pd.read_csv(path, encoding='utf-8-sig', engine='python')

# -------------------------------------------------------------
# 2. [핵심 로직] 단속 데이터에서 '순수 평일' 날짜만 쏙 뽑아내기
# -------------------------------------------------------------
df_enf = load_csv_safe("enforcement_with_grid_id.csv")

# enforced_at (예: 2025-01-01 14:00) 에서 날짜(YYYY-MM-DD)만 추출
df_enf['date_only'] = df_enf['enforced_at'].astype(str).str.split(' ').str[0]

# 주말과 공휴일/명절을 철저히 배제한 '평일'의 날짜 리스트 확보
working_days = df_enf[df_enf['day_type'] == '평일']['date_only'].unique()

# -------------------------------------------------------------
# 3. 대기질 데이터 로드 및 '순수 평일' 필터링
# -------------------------------------------------------------
df_air = load_csv_safe("air_quality_transformed (1).csv")

df_air['clean_at'] = df_air['measured_at'].astype(str).str.replace(' 24:', ' 00:')
df_air['dt'] = pd.to_datetime(df_air['clean_at'])
df_air['date_only'] = df_air['dt'].dt.strftime('%Y-%m-%d')

# ★ 대기질 데이터도 단속 데이터 기준 일하는 '평일'만 남기도록 강력 필터링!
df_air_weekday = df_air[df_air['date_only'].isin(working_days)].copy()

# 4. 프론트엔드 선택 옵션용 [월, 요일, 시간대] 컬럼 생성
df_air_weekday['month'] = df_air_weekday['dt'].dt.month
dow_map = {0:'월', 1:'화', 2:'수', 3:'목', 4:'금', 5:'토', 6:'일'} # 사실상 월~금만 남게 됨
df_air_weekday['day_of_week'] = df_air_weekday['dt'].dt.dayofweek.map(dow_map)
df_air_weekday['hour'] = df_air_weekday['dt'].dt.hour

# -------------------------------------------------------------
# 5. 가중치 보정 (도로변 vs 도시대기) - 오직 평일 데이터로만 산출
# -------------------------------------------------------------
road_df = df_air_weekday[df_air_weekday['station_name'] == '대왕판교로']
bg_df = df_air_weekday[df_air_weekday['station_name'] != '대왕판교로']

grp_cols = ['month', 'day_of_week', 'hour']
pollutants = ['no2', 'co', 'pm10', 'pm25']

road_mean = road_df.groupby(grp_cols)[pollutants].mean().reset_index()
bg_mean = bg_df.groupby(grp_cols)[pollutants].mean().reset_index()

# 월/요일/시간대별 가중치 병합
weights = pd.merge(road_mean, bg_mean, on=grp_cols, suffixes=('_road', '_bg'))

for p in pollutants:
    # 예외 처리: 주거지 수치가 0일 경우 에러 방지 (기본 가중치 1.0 부여)
    weights[f'weight_{p}'] = np.where(
        weights[f'{p}_bg'] > 0,
        weights[f'{p}_road'] / weights[f'{p}_bg'],
        1.0
    )

# -------------------------------------------------------------
# 6. 원본 데이터에 가중치 매핑 및 최종 수치 보정
# -------------------------------------------------------------
df_merged = pd.merge(df_air_weekday, weights[grp_cols + [f'weight_{p}' for p in pollutants]], on=grp_cols, how='left')

for p in pollutants:
    # 도로변(대왕판교로)은 실제 수치 유지, 도시대기는 평일 가중치를 곱해 현실화
    df_merged[p] = np.where(
        df_merged['station_name'] == '대왕판교로',
        df_merged[p],
        df_merged[p] * df_merged[f'weight_{p}']
    )

# 7. 최종 파일 정리 및 저장 (불필요한 계산용 컬럼 제거)
cols_to_drop = [col for col in df_merged.columns if col.startswith('weight_')] + ['clean_at', 'dt', 'date_only']
df_final = df_merged.drop(columns=cols_to_drop)

output_file = "air_quality_corrected_weekday_only.csv"
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 완벽한 '출근일(평일)' 전용 데이터 변환 완료! '{output_file}'이 생성되었습니다.")

🎉 완벽한 '출근일(평일)' 전용 데이터 변환 완료! 'air_quality_corrected_weekday_only.csv'이 생성되었습니다.


## 부천시

In [13]:
import pandas as pd
import numpy as np

# 1. 파일 로드 및 초기 병합
files = [
    "중2동(도시)_2025_1년통합본.csv",
    "송내대로(도시)_2025_1년통합본.csv",
    "부천소사본동(도로)_2025_1년통합본.csv",
    "부천내동(도로)_2025_1년통합본.csv"
]

# 관측소별 대략적 위경도 맵핑 (필요시 실제 정확한 좌표로 교체)
station_meta = {
    '중2동': {'lat': 37.4939079, 'lng': 126.7700835},
    '송내대로': {'lat': 37.5060355, 'lng': 126.7595460},
    '부천소사본동': {'lat': 37.4800814, 'lng': 126.7999303},
    '부천내동': {'lat': 37.5201728, 'lng': 126.7733045}
}

df_list = []
for f in files:
    station_name = f.split('(')[0]
    try:
        temp_df = pd.read_csv(f, encoding='cp949')
    except:
        temp_df = pd.read_csv(f, encoding='utf-8')

    temp_df['station_name'] = station_name
    df_list.append(temp_df)

df_air = pd.concat(df_list, ignore_index=True)

# 2. 날짜/시간 파싱 및 정렬 (보간법을 위해 시간순 정렬 필수)
def parse_datetime(dt_str):
    date_part, hour_part = dt_str.split(':')
    if hour_part == '24':
        return pd.to_datetime(date_part) + pd.Timedelta(days=1)
    else:
        return pd.to_datetime(f"{date_part} {hour_part}:00:00")

df_air['dt'] = df_air['측정일시'].apply(parse_datetime)
# 관측소별, 시간순으로 완벽하게 정렬
df_air = df_air.sort_values(by=['station_name', 'dt']).reset_index(drop=True)

df_air['date_only'] = df_air['dt'].dt.strftime('%Y-%m-%d')
df_air['month'] = df_air['dt'].dt.month
df_air['hour'] = df_air['dt'].dt.hour
dow_map = {0:'월', 1:'화', 2:'수', 3:'목', 4:'금', 5:'토', 6:'일'}
df_air['day_of_week'] = df_air['dt'].dt.dayofweek.map(dow_map)

pollutants = ['NO2', 'CO']
# 데이터 타입을 명확히 숫자로 변환 (결측치는 NaN으로 처리)
for p in pollutants:
    df_air[p] = pd.to_numeric(df_air[p], errors='coerce')

# =========================================================================
# 🌟 [핵심] 3단계 결측치 방어 로직 (시뮬레이션 블랙홀 완벽 차단)
# =========================================================================
print("🛠️ 결측치(NaN) 3단계 방어 로직 처리를 시작합니다...")

# [1단계] 1~2시간 짧은 누락 ➔ 선형 보간 (Linear Interpolation)
# 동일 관측소 내에서 앞뒤 시간대의 데이터를 선으로 이어 자연스럽게 채움 (최대 2시간 연속 결측치까지만)
for p in pollutants:
    df_air[p] = df_air.groupby('station_name')[p].transform(
        lambda x: x.interpolate(method='linear', limit=2, limit_direction='both')
    )

# [2단계] 반나절~하루 누락 ➔ 자가 평균 대치 (Historical Average)
# 1단계로 못 채운 구멍은 '동일 관측소 + 같은 월 + 같은 요일 + 같은 시간대'의 평균으로 대치
self_mean = df_air.groupby(['station_name', 'month', 'day_of_week', 'hour'])[pollutants].transform('mean')
for p in pollutants:
    df_air[p] = df_air[p].fillna(self_mean[p])

# [3단계] 장기 누락 ➔ 부천시 내 타 지역 관측소 동시간대 평균
# 센서 장기 고장 등으로 1, 2단계를 다 거치고도 비어있는 초장기 결측치는 타 지역 평균으로 대치
other_mean = df_air.groupby(['month', 'day_of_week', 'hour'])[pollutants].transform('mean')
for p in pollutants:
    df_air[p] = df_air[p].fillna(other_mean[p])

# (안전망) 1~3단계를 거치고도 혹시 남은 값이 있다면 전체 평균으로 최후 방어
for p in pollutants:
    df_air[p] = df_air[p].fillna(df_air[p].mean())

print("✅ 결측치 보완 100% 완료!")
# =========================================================================

# 3. '순수 평일' 필터링 (주말 및 2025 법정 공휴일 제거)
holidays_2025 = ['2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',
                 '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06',
                 '2025-08-15', '2025-10-03', '2025-10-06', '2025-10-07', '2025-10-08',
                 '2025-10-09', '2025-12-25']

df_air_weekday = df_air[
    (df_air['dt'].dt.dayofweek <= 4) &
    (~df_air['date_only'].isin(holidays_2025))
].copy()

# 4. [지역별 × 월별 × 시간대별] 3차원 가중치 연산
road_df = df_air_weekday[df_air_weekday['station_name'] == '부천내동']
bg_df = df_air_weekday[df_air_weekday['station_name'] != '부천내동']

grp_cols_road = ['month', 'hour']
grp_cols_bg = ['station_name', 'month', 'hour']

road_mean = road_df.groupby(grp_cols_road)[pollutants].mean().reset_index()
bg_mean = bg_df.groupby(grp_cols_bg)[pollutants].mean().reset_index()

weights = pd.merge(bg_mean, road_mean, on=['month', 'hour'], suffixes=('_bg', '_road'))

for p in pollutants:
    weights[f'weight_{p}'] = np.where(
        weights[f'{p}_bg'] > 0,
        weights[f'{p}_road'] / weights[f'{p}_bg'],
        1.0
    )

# 5. 원본 데이터에 가중치 매핑 및 최종 수치 현실화
df_merged = pd.merge(df_air_weekday,
                     weights[['station_name', 'month', 'hour'] + [f'weight_{p}' for p in pollutants]],
                     on=['station_name', 'month', 'hour'],
                     how='left')

for p in pollutants:
    # 도로변(부천내동)은 가중치가 없으므로 1.0으로 처리하여 원본 값 유지
    df_merged[f'weight_{p}'] = df_merged[f'weight_{p}'].fillna(1.0)
    df_merged[f'adj_{p}'] = df_merged[p] * df_merged[f'weight_{p}']

# 6. 최종 프론트엔드/백엔드 DB용 ERD 규격 맞춤
df_merged['air_id'] = range(1, len(df_merged) + 1)
df_merged['grid_id'] = np.nan
df_merged['lat'] = df_merged['station_name'].map(lambda x: station_meta[x]['lat'])
df_merged['lng'] = df_merged['station_name'].map(lambda x: station_meta[x]['lng'])
df_merged['measured_at'] = df_merged['dt'].dt.strftime('%Y-%m-%d %H:%M')

df_merged['no2'] = df_merged['adj_NO2'].round(4)
df_merged['co'] = df_merged['adj_CO'].round(4)

# 이미지의 컬럼 순서 일치
final_columns = ['air_id', 'grid_id', 'station_name', 'lat', 'lng', 'measured_at', 'month', 'day_of_week', 'hour', 'no2', 'co']
df_final = df_merged[final_columns].sort_values(by=['measured_at', 'station_name'])

# 7. 결과 산출
output_file = "bucheon_air_quality_corrected_erd.csv"
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 부천시 데이터 파이프라인 완벽 가동: {output_file}")

🛠️ 결측치(NaN) 3단계 방어 로직 처리를 시작합니다...
✅ 결측치 보완 100% 완료!
🎉 부천시 데이터 파이프라인 완벽 가동: bucheon_air_quality_corrected_erd.csv


## grid_id와 결합

In [7]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 파일 로드 (경로는 코랩 환경에 맞게 수정해 주세요)
air_path = '/content/drive/MyDrive/bucheon_air_quality_corrected_erd.csv'
grid_path = '/content/drive/MyDrive/grids_bucheon_final.csv'

def load_csv_safe(path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    return pd.read_csv(path)

df_air = load_csv_safe(air_path)
df_grid = load_csv_safe(grid_path)

print(f"☁️ 대기질 데이터: {len(df_air):,}건, 격자 데이터: {len(df_grid):,}건 로드 완료")

# 2. 그리드 중심 좌표 기준 KD-Tree 구성
grid_coords = df_grid[['center_lat', 'center_lng']].values
tree = cKDTree(grid_coords)

# 3. 대기질 관측소 좌표를 기반으로 가장 가까운 격자 탐색
air_coords = df_air[['lat', 'lng']].values
distances, indices = tree.query(air_coords)

# 4. 찾은 인덱스를 바탕으로 grid_id 업데이트 (정수형 변환)
df_air['grid_id'] = df_grid.iloc[indices]['grid_id'].values.astype(int)

# 5. 최종 변환된 CSV 저장 (백엔드 적재용 UTF-8-SIG)
output_file = '/content/drive/MyDrive/bucheon_air_quality_grid_mapped.csv'
df_air.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n🎉 대기질 데이터 격자 매핑 완료!")
print(f"👉 파일 저장 위치: {output_file}")
print(df_air[['air_id', 'grid_id', 'station_name', 'lat', 'lng', 'measured_at']].head(5))

☁️ 대기질 데이터: 23,428건, 격자 데이터: 55,256건 로드 완료

🎉 대기질 데이터 격자 매핑 완료!
👉 파일 저장 위치: /content/drive/MyDrive/bucheon_air_quality_grid_mapped.csv
   air_id  grid_id station_name        lat         lng       measured_at
0       1    46474         부천내동  37.520173  126.773304  2025-01-02 00:00
1    5858    15863       부천소사본동  37.480081  126.799930  2025-01-02 00:00
2   11715    35576         송내대로  37.506036  126.759546  2025-01-02 00:00
3   17572    26732          중2동  37.493908  126.770083  2025-01-02 00:00
4       2    46474         부천내동  37.520173  126.773304  2025-01-02 01:00


# 그리드 테이블 생성


In [ ]:
import geopandas as gpd
import pandas as pd

# 1. 100M 격자 파일 읽기
gdf = gpd.read_file('grid_다사_100M.shp')

# 2. 카카오/네이버 지도 좌표계(WGS84, EPSG:4326)로 변환
gdf_4326 = gdf.to_crs(epsg=4326)

# 3. Bounding Box (min_lng, min_lat, max_lng, max_lat) 추출
bounds = gdf_4326.bounds

# 4. grids 테이블 스키마에 맞게 컬럼 가공
grids_df = pd.DataFrame()
grids_df['grid_id'] = range(1, len(gdf_4326) + 1)
grids_df['district_id'] = 1  # 분당구 ID
grids_df['grid_code'] = gdf_4326['GRID_CD']

grids_df['min_lat'] = bounds['miny']
grids_df['min_lng'] = bounds['minx']
grids_df['max_lat'] = bounds['maxy']
grids_df['max_lng'] = bounds['maxx']

# 중심점 추출
centroids = gdf_4326.geometry.centroid
grids_df['center_lat'] = centroids.y
grids_df['center_lng'] = centroids.x

grids_df['area_km2'] = 0.01  # 100m x 100m = 0.01 km^2
grids_df['effective_area_km2'] = 0.01

# DB 적재용 CSV 변환 완료
grids_df.to_csv('grids_table_ready.csv', index=False, encoding='utf-8-sig')

/tmp/ipykernel_3053/3676194054.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf_4326.geometry.centroid


## 분당구 내 그리드 필터링
단속데이터 위경도 기준 제일 낮은 데이터 소수점 둘째자리 기준 최소치는 내림, 최대치는 올림법 적용.

In [ ]:
import pandas as pd

# 1. 파일 읽기
df = pd.read_csv('grids_table_ready.csv')

# 2. 좌표 결측치(NaN) 제거
df_clean = df.dropna(subset=['center_lat', 'center_lng']).copy()

# 3. 분당구 Bounding Box 영역만 필터링 (분당구 실제 위경도 범위)
bundang_grids = df_clean[
    (df_clean['center_lat'] >= 37.27) & (df_clean['center_lat'] <= 37.50) &
    (df_clean['center_lng'] >= 126.52) & (df_clean['center_lng'] <= 127.18)
].copy()

# 4. grid_id 재정렬 (1부터 다시 부여)
bundang_grids['grid_id'] = range(1, len(bundang_grids) + 1)

# 5. 최종 파일 저장
bundang_grids.to_csv('grids_bundang_final.csv', index=False, encoding='utf-8-sig')

print(f"정제 완료! 총 {len(bundang_grids):,}개의 분당구 격자 데이터가 생성되었습니다.")

정제 완료! 총 122,318개의 분당구 격자 데이터가 생성되었습니다.


## 부천시내 그리드 필터링

In [14]:
import pandas as pd

# 1. 파일 읽기
df = pd.read_csv('grids_table_ready.csv')

# 2. 좌표 결측치(NaN) 제거
df_clean = df.dropna(subset=['center_lat', 'center_lng']).copy()

# 3. 부천시 Bounding Box 영역만 필터링 (부천시 실제 위경도 범위)
bundang_grids = df_clean[
    (df_clean['center_lat'] >= 37.46) & (df_clean['center_lat'] <= 37.56) &
    (df_clean['center_lng'] >= 126.73) & (df_clean['center_lng'] <= 127.84)
].copy()

# 4. grid_id 재정렬 (1부터 다시 부여)
bundang_grids['grid_id'] = range(1, len(bundang_grids) + 1)

# 5. 최종 파일 저장
bundang_grids.to_csv('grids_bucheon_final.csv', index=False, encoding='utf-8-sig')

print(f"정제 완료! 총 {len(bundang_grids):,}개의 부천시 격자 데이터가 생성되었습니다.")

정제 완료! 총 55,256개의 부천시 격자 데이터가 생성되었습니다.


# risk_index 테이블 생성

## 성남시

In [12]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 인코딩 및 에러 방지 데이터 로드 함수
def load_csv_safe(file_path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc, engine='c')
            print(f"✅ [{file_path}] 읽기 성공")
            return df
        except:
            continue
    return pd.read_csv(file_path, encoding='utf-8-sig', engine='python')

# 2. 분당구 4대 핵심 데이터 로드 (경로 유지)
df_grid = load_csv_safe("/content/drive/MyDrive/grids_bundang_final (1).csv")
df_enf = load_csv_safe("/content/drive/MyDrive/enforcement_with_grid_id_fixed.csv")
df_air = load_csv_safe("/content/drive/MyDrive/air_quality_summary_oracle (1).csv")
df_apt = load_csv_safe("/content/drive/MyDrive/apartments_with_grid_id.csv")

def min_max(series):
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return series * 0.0
    return (series - s_min) / (s_max - s_min + 1e-6)

# =====================================================================
# 🌟 [최적화 1] 단속 또는 아파트가 존재하는 '활성 격자(Active Grid)' 필터링
# =====================================================================
active_grid_ids = pd.concat([df_enf['grid_id'], df_apt['grid_id']]).dropna().unique()
df_active_grids = df_grid[df_grid['grid_id'].isin(active_grid_ids)].copy()
print(f"🎯 전체 {len(df_grid):,}개 격자 중, 분석 대상인 활성 격자 {len(df_active_grids):,}개 추출 완료!")

# =====================================================================
# 🌟 [최종 기획 반영] 수요(단속)와 공급(아파트 유휴면)을 동시 고려한 지수 산출
# =====================================================================
# 1. 활성 격자별 단속 건수 (수요)
enf_grid_counts = df_enf.groupby('grid_id').size().reset_index(name='enf_count')

# 2. 활성 격자별 아파트 유휴 주차면 합계 (공급)
apt_open_counts = df_apt.groupby('grid_id')['open_count'].sum().reset_index(name='total_open_count')

# 3. Base 테이블 병합
df_base = df_active_grids[['grid_id', 'center_lat', 'center_lng']].merge(
    enf_grid_counts, on='grid_id', how='left'
).merge(
    apt_open_counts, on='grid_id', how='left'
).fillna({'enf_count': 0, 'total_open_count': 0})

# 4. 수요 압박 및 공급 부족 지수 산출 (Log 변환 및 역수 적용)
df_base['demand_pressure'] = min_max(np.log1p(df_base['enf_count']))
df_base['supply_shortage'] = 1.0 - min_max(df_base['total_open_count']) # 빈자리가 적을수록 1.0에 근접

# =====================================================================
# 🌟 [오류 수정 및 최적화 2] 24시 표기 오류 완벽 해결 및 측정소 매핑 (cKDTree)
# =====================================================================
print("🔄 대기질 시간 파싱 및 최단 거리 측정소 매핑 중...")

# [에러 픽스] 24:00:00 을 00:00:00 으로 치환하여 파싱 에러 원천 차단
df_air['measured_at_fixed'] = df_air['measured_at'].astype(str).str.replace(' 24:00:00', ' 00:00:00').str.replace(' 24:00', ' 00:00')
df_air['hour'] = pd.to_datetime(df_air['measured_at_fixed']).dt.hour

air_grid_h = df_air.groupby(['grid_id', 'hour'])[['no2', 'co']].mean().reset_index()
air_overall_h = df_air.groupby('hour')[['no2', 'co']].mean().reset_index()

station_info = df_air[['grid_id', 'lat', 'lng']].drop_duplicates()
# 좌표 결측치 방어
station_info = station_info.dropna(subset=['lat', 'lng'])
tree = cKDTree(station_info[['lat', 'lng']].values)

distances, indices = tree.query(df_base[['center_lat', 'center_lng']].values)
df_base['nearest_station_grid_id'] = station_info['grid_id'].values[indices]

# =====================================================================
# 🌟 [최적화 3] Vectorization 교차 조인 및 최종 위험 지수(Risk Score) 산출
# =====================================================================
print("🔄 위험지수(Risk Index) 24시간 매트릭스 일괄 연산 중...")
traffic_weights = pd.DataFrame(list({
    0:0.7, 1:0.7, 2:0.7, 3:0.7, 4:0.7, 5:0.7, 6:0.8,
    7:1.3, 8:1.3, 9:1.3, 10:1.0, 11:1.0, 12:1.0, 13:1.0, 14:1.0, 15:1.0, 16:1.0,
    17:1.3, 18:1.3, 19:1.3, 20:1.3, 21:0.8, 22:0.7, 23:0.7
}.items()), columns=['hour', 'tw'])

# 1. 24시간 확장 (Cross Join)
df_expanded = df_base.merge(pd.DataFrame({'hour': range(24)}), how='cross')
df_expanded = df_expanded.merge(traffic_weights, on='hour', how='left')

# 2. 교통 혼잡도 (TC) 계산
df_expanded['tc'] = np.minimum(1.0, df_expanded['demand_pressure'] * (df_expanded['tw'] / 1.3))

# 3. 내 격자에 매핑된 측정소의 대기질 병합 및 결측치 이중 방어
df_expanded = df_expanded.merge(
    air_grid_h,
    left_on=['nearest_station_grid_id', 'hour'],
    right_on=['grid_id', 'hour'],
    how='left',
    suffixes=('', '_air')
)
df_expanded = df_expanded.merge(air_overall_h, on='hour', how='left', suffixes=('', '_overall'))
df_expanded['no2'] = df_expanded['no2'].fillna(df_expanded['no2_overall'])
df_expanded['co'] = df_expanded['co'].fillna(df_expanded['co_overall'])

# 4. 환경 민감도 (ES) 및 최종 Risk Score 계산
df_expanded['es'] = np.minimum(1.0, (df_expanded['no2'] / 0.075) * 0.6 + (df_expanded['co'] / 1.79) * 0.4)
df_expanded['risk_score'] = (0.35 * df_expanded['supply_shortage'] +
                             0.25 * df_expanded['demand_pressure'] +
                             0.15 * df_expanded['tc'] +
                             0.25 * df_expanded['es']) * 100.0

# =====================================================================
# 5. 최종 데이터 프레임 포맷팅 및 추출
# =====================================================================
batch_date_str = pd.Timestamp.now(tz='Asia/Seoul').strftime('%Y-%m-%d')

df_risk_final = pd.DataFrame({
    'grid_id': df_expanded['grid_id'],
    'hour_of_day': df_expanded['hour'],
    'demand_pressure': df_expanded['demand_pressure'].round(4),
    'supply_shortage': df_expanded['supply_shortage'].round(4),
    'traffic_congest': df_expanded['tc'].round(4),
    'env_sensitivity': df_expanded['es'].round(4),
    'risk_score': df_expanded['risk_score'].round(2),
    'batch_date': batch_date_str
})

df_risk_final.insert(0, 'risk_id', range(1, len(df_risk_final) + 1))
output_file = "/content/drive/MyDrive/bundang_risk_index_final_v2.csv"
df_risk_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 에러 완벽 해결! 분당구 B2G 시뮬레이션 지수 산출 완료! ({output_file})")

✅ [/content/drive/MyDrive/grids_bundang_final (1).csv] 읽기 성공
✅ [/content/drive/MyDrive/enforcement_with_grid_id_fixed.csv] 읽기 성공
✅ [/content/drive/MyDrive/air_quality_summary_oracle (1).csv] 읽기 성공
✅ [/content/drive/MyDrive/apartments_with_grid_id.csv] 읽기 성공
🎯 전체 122,318개 격자 중, 분석 대상인 활성 격자 1,306개 추출 완료!
🔄 대기질 시간 파싱 및 최단 거리 측정소 매핑 중...
🔄 위험지수(Risk Index) 24시간 매트릭스 일괄 연산 중...
🎉 에러 완벽 해결! 분당구 B2G 시뮬레이션 지수 산출 완료! (/content/drive/MyDrive/bundang_risk_index_final_v2.csv)


## 부천시

In [11]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 인코딩 및 에러 방지 데이터 로드 함수
def load_csv_safe(file_path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc, engine='c')
            print(f"✅ [{file_path}] 읽기 성공")
            return df
        except:
            continue
    return pd.read_csv(file_path, encoding='utf-8-sig', engine='python')

# 2. 방금 전까지 완벽하게 정제한 부천시 4대 핵심 데이터 로드
df_grid = load_csv_safe("/content/drive/MyDrive/grids_bucheon_final.csv") # 부천시 최종 그리드
df_enf = load_csv_safe("/content/drive/MyDrive/부천시_단속위치_grid_mapped.csv") # 부천시 단속 데이터
df_apt = load_csv_safe("/content/drive/MyDrive/bucheon_apartments_erd.csv") # 부천시 아파트 데이터
df_air = load_csv_safe("/content/drive/MyDrive/bucheon_air_quality_grid_mapped.csv") # 부천시 대기질 데이터

def min_max(series):
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return series * 0.0
    return (series - s_min) / (s_max - s_min + 1e-6)

# =====================================================================
# 🌟 [최적화 1] 단속 또는 아파트가 존재하는 '활성 격자(Active Grid)' 필터링
# =====================================================================
active_grid_ids = pd.concat([df_enf['grid_id'], df_apt['grid_id']]).dropna().unique()
df_active_grids = df_grid[df_grid['grid_id'].isin(active_grid_ids)].copy()
print(f"🎯 전체 {len(df_grid):,}개 격자 중, 분석 대상인 활성 격자 {len(df_active_grids):,}개 추출 완료!")

# =====================================================================
# 🌟 [최종 기획 반영] 수요(단속)와 공급(아파트 유휴면)을 동시 고려한 지수 산출
# =====================================================================
# 1. 활성 격자별 단속 건수 (수요)
enf_grid_counts = df_enf.groupby('grid_id').size().reset_index(name='enf_count')

# 2. 활성 격자별 아파트 유휴 주차면 합계 (공급)
apt_open_counts = df_apt.groupby('grid_id')['open_count'].sum().reset_index(name='total_open_count')

# 3. Base 테이블 병합
df_base = df_active_grids[['grid_id', 'center_lat', 'center_lng']].merge(
    enf_grid_counts, on='grid_id', how='left'
).merge(
    apt_open_counts, on='grid_id', how='left'
).fillna({'enf_count': 0, 'total_open_count': 0})

# 4. 수요 압박 및 공급 부족 지수 산출 (Log 변환 및 역수 적용)
df_base['demand_pressure'] = min_max(np.log1p(df_base['enf_count']))
df_base['supply_shortage'] = 1.0 - min_max(df_base['total_open_count']) # 빈자리가 적을수록 1.0에 근접

# =====================================================================
# 🌟 [최적화 2] 1시간 단위 대기질 시간 파싱 및 가장 가까운 측정소 매핑 (cKDTree)
# =====================================================================
# measured_at 포맷: "2025-01-02 01:00" -> 시간만 추출
df_air['hour'] = pd.to_datetime(df_air['measured_at']).dt.hour
air_grid_h = df_air.groupby(['grid_id', 'hour'])[['no2', 'co']].mean().reset_index()
air_overall_h = df_air.groupby('hour')[['no2', 'co']].mean().reset_index()

print("🔄 활성 격자에 가장 가까운 대기질 측정소를 매핑합니다...")
station_info = df_air[['grid_id', 'lat', 'lng']].drop_duplicates()
tree = cKDTree(station_info[['lat', 'lng']].values)

distances, indices = tree.query(df_base[['center_lat', 'center_lng']].values)
df_base['nearest_station_grid_id'] = station_info['grid_id'].values[indices]

# =====================================================================
# 🌟 [최적화 3] Vectorization 교차 조인 및 최종 위험 지수(Risk Score) 산출
# =====================================================================
print("🔄 위험지수(Risk Index) 24시간 매트릭스 연산 중...")
traffic_weights = pd.DataFrame(list({
    0:0.7, 1:0.7, 2:0.7, 3:0.7, 4:0.7, 5:0.7, 6:0.8,
    7:1.3, 8:1.3, 9:1.3, 10:1.0, 11:1.0, 12:1.0, 13:1.0, 14:1.0, 15:1.0, 16:1.0,
    17:1.3, 18:1.3, 19:1.3, 20:1.3, 21:0.8, 22:0.7, 23:0.7
}.items()), columns=['hour', 'tw'])

# 1. 24시간 확장 (Cross Join)
df_expanded = df_base.merge(pd.DataFrame({'hour': range(24)}), how='cross')
df_expanded = df_expanded.merge(traffic_weights, on='hour', how='left')

# 2. 교통 혼잡도 (TC) 계산
df_expanded['tc'] = np.minimum(1.0, df_expanded['demand_pressure'] * (df_expanded['tw'] / 1.3))

# 3. 내 격자에 매핑된 측정소의 대기질 병합 및 결측치 이중 방어
df_expanded = df_expanded.merge(
    air_grid_h,
    left_on=['nearest_station_grid_id', 'hour'],
    right_on=['grid_id', 'hour'],
    how='left',
    suffixes=('', '_air')
)
df_expanded = df_expanded.merge(air_overall_h, on='hour', how='left', suffixes=('', '_overall'))
df_expanded['no2'] = df_expanded['no2'].fillna(df_expanded['no2_overall'])
df_expanded['co'] = df_expanded['co'].fillna(df_expanded['co_overall'])

# 4. 환경 민감도 (ES) 및 최종 Risk Score 계산
df_expanded['es'] = np.minimum(1.0, (df_expanded['no2'] / 0.075) * 0.6 + (df_expanded['co'] / 1.79) * 0.4)
df_expanded['risk_score'] = (0.35 * df_expanded['supply_shortage'] +
                             0.25 * df_expanded['demand_pressure'] +
                             0.15 * df_expanded['tc'] +
                             0.25 * df_expanded['es']) * 100.0

# =====================================================================
# 5. 최종 데이터 프레임 포맷팅 및 추출
# =====================================================================
batch_date_str = pd.Timestamp.now(tz='Asia/Seoul').strftime('%Y-%m-%d')

df_risk_final = pd.DataFrame({
    'grid_id': df_expanded['grid_id'],
    'hour_of_day': df_expanded['hour'],
    'demand_pressure': df_expanded['demand_pressure'].round(4),
    'supply_shortage': df_expanded['supply_shortage'].round(4),
    'traffic_congest': df_expanded['tc'].round(4),
    'env_sensitivity': df_expanded['es'].round(4),
    'risk_score': df_expanded['risk_score'].round(2),
    'batch_date': batch_date_str
})

df_risk_final.insert(0, 'risk_id', range(1, len(df_risk_final) + 1))
output_file = "/content/drive/MyDrive/bucheon_risk_index_final.csv"
df_risk_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 부천시 타겟 B2G 시뮬레이션 지수 산출 완료! ({output_file})")

✅ [/content/drive/MyDrive/grids_bucheon_final.csv] 읽기 성공
✅ [/content/drive/MyDrive/부천시_단속위치_grid_mapped.csv] 읽기 성공
✅ [/content/drive/MyDrive/bucheon_apartments_erd.csv] 읽기 성공
✅ [/content/drive/MyDrive/bucheon_air_quality_grid_mapped.csv] 읽기 성공
🎯 전체 55,256개 격자 중, 분석 대상인 활성 격자 2,264개 추출 완료!
🔄 활성 격자에 가장 가까운 대기질 측정소를 매핑합니다...
🔄 위험지수(Risk Index) 24시간 매트릭스 연산 중...
🎉 부천시 타겟 B2G 시뮬레이션 지수 산출 완료! (/content/drive/MyDrive/bucheon_risk_index_final.csv)


#1. 이슈 수정 누락 데이터 구제 (경도 위도 오삽입 데이터 구제)

In [ ]:
import pandas as pd

# 1. 파일 불러오기
file_relief = "분당구_단속위치_최종_구제완료.csv"  # 이전 구제완료 파일
file_work = "분당구_단속위치_경도위도 작업본.csv"    # 최신 작업본 파일

# 인코딩 설정에 맞춰 읽기
df_relief = pd.read_csv(file_relief, encoding='utf-8')
df_work = pd.read_csv(file_work, encoding='cp949')

# 작업본 복사본 생성 (원본 유지)
df_master = df_work.copy()

# ---------------------------------------------------------------------------
# [STEP 1] 동일 행 인덱스(Index) 기준 1차 수복
# ---------------------------------------------------------------------------
# 작업본에서 이상 좌표(위도 > 38 또는 Null)인 행 식별
abnormal_mask_work = (df_master['latitude'] > 38) | (df_master['latitude'].isnull())

# 이전 구제완료 파일에서 해당 인덱스의 좌표가 정상(위도 33~38, 경도 124~132)인 조건
valid_mask_relief = (
    (df_relief['latitude'] >= 33) & (df_relief['latitude'] <= 38) &
    (df_relief['longitude'] >= 124) & (df_relief['longitude'] <= 132)
)

# 동일 인덱스에 대해 구제완료 파일의 정상 좌표 덮어씌우기
index_fix_mask = abnormal_mask_work & valid_mask_relief
df_master.loc[index_fix_mask, 'latitude'] = df_relief.loc[index_fix_mask, 'latitude']
df_master.loc[index_fix_mask, 'longitude'] = df_relief.loc[index_fix_mask, 'longitude']

# ---------------------------------------------------------------------------
# [STEP 2] latitude > 38 (경도값 127.xxx가 위도에 잘못 들어간 건) 보정
# ---------------------------------------------------------------------------
swapped_mask = df_master['latitude'] > 38

# latitude에 있던 127.xxx 값을 longitude로 이동하고 latitude는 임시 NaN 처리
df_master.loc[swapped_mask, 'longitude'] = df_master.loc[swapped_mask, 'latitude']
df_master.loc[swapped_mask, 'latitude'] = None

# ---------------------------------------------------------------------------
# [STEP 3] '단속장소' 명칭 기반 교차 매핑 (2차 수복)
# ---------------------------------------------------------------------------
# 공백/따옴표 제거한 정제용 장소 칼럼 생성
df_master['clean_place'] = df_master['단속장소'].astype(str).str.strip().str.replace("'", "")

# 전체 데이터셋 중 '정상 좌표'를 가진 행들만 추출하여 룩업 매핑 테이블 구축
valid_master = df_master[
    (df_master['latitude'] >= 33) & (df_master['latitude'] <= 38) &
    (df_master['longitude'] >= 124) & (df_master['longitude'] <= 132)
].dropna(subset=['latitude', 'longitude'])

place_coord_map = valid_master.groupby('clean_place')[['latitude', 'longitude']].first().to_dict(orient='index')

# latitude가 여전히 비어있는(NaN) 행들에 대해 단속장소 이름으로 좌표 채우기
missing_lat_mask = df_master['latitude'].isnull()

for idx in df_master[missing_lat_mask].index:
    c_place = df_master.at[idx, 'clean_place']
    if c_place in place_coord_map:
        df_master.at[idx, 'latitude'] = place_coord_map[c_place]['latitude']
        df_master.at[idx, 'longitude'] = place_coord_map[c_place]['longitude']

# 임시 칼럼 삭제
df_master = df_master.drop(columns=['clean_place'])

# ---------------------------------------------------------------------------
# [STEP 4] 검증 및 최종 저장
# ---------------------------------------------------------------------------
gt_38_count = (df_master['latitude'] > 38).sum()
null_lat_count = df_master['latitude'].isnull().sum()

print("===== 최종 검증 결과 =====")
print(f"1. 위도 > 38 이상치 건수: {gt_38_count}건")
print(f"2. 남은 결측치(Null) 건수: {null_lat_count}건")

# UTF-8 BOM 인코딩으로 저장 (엑셀/GIS 호환)
output_filename = "분당구_단속위치_최종_보정완료.csv"
df_master.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"성공적으로 '{output_filename}' 파일 생성이 완료되었습니다!")

===== 최종 검증 결과 =====
1. 위도 > 38 이상치 건수: 0건
2. 남은 결측치(Null) 건수: 739건
성공적으로 '분당구_단속위치_최종_보정완료.csv' 파일 생성이 완료되었습니다!


## 분당 오류 데이터 수정

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

# 1. 파일 로드 (인코딩 에러 방지)
def load_csv_safe(path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, engine='c')
        except:
            pass
    return pd.read_csv(path, encoding='utf-8-sig', engine='python')

# 현재 작업 중인 단속 데이터와 분당구 최종 격자 데이터 로드
df_enf = load_csv_safe("enforcement_weekday_only (1).csv")
df_grids = load_csv_safe("grids_bundang_final (1).csv")

# 2. [라스트 마일] 오인된 3곳의 올바른 좌표 수동 덮어쓰기
manual_coords = {
    '운중로14번길 주택단지 부근': (37.3904231, 127.0659508),
    '장안로41번길 부근': (37.3706476, 127.1420296),
    '벌말로42': (37.4114359, 127.1423436)
}

updated_cnt = 0
for place, (lat, lng) in manual_coords.items():
    # 정확히 일치하는 단속 장소명만 타겟팅
    mask = df_enf['place_text'].astype(str).str.strip() == place

    # 좌표 업데이트
    df_enf.loc[mask, 'lat'] = lat
    df_enf.loc[mask, 'lng'] = lng
    updated_cnt += mask.sum()

print(f"✅ 총 {updated_cnt}건의 잘못된 좌표가 정상적으로 수정되었습니다.")

# 3. Geopandas를 활용한 Grid ID 공간 재매핑 (Spatial Join)
print("🔄 수정된 좌표를 바탕으로 올바른 격자(Grid)를 다시 찾고 있습니다...")

# Bounding Box를 활용해 100M 격자 폴리곤 생성
df_grids['geometry'] = df_grids.apply(
    lambda r: box(r['min_lng'], r['min_lat'], r['max_lng'], r['max_lat']), axis=1
)
gdf_grids = gpd.GeoDataFrame(df_grids, geometry='geometry', crs="EPSG:4326")

# 단속 데이터를 포인트(Point)로 변환
gdf_enf = gpd.GeoDataFrame(
    df_enf,
    geometry=gpd.points_from_xy(df_enf['lng'], df_enf['lat']),
    crs="EPSG:4326"
)

# 기존에 잘못 들어간 grid_id 열 제거
if 'grid_id' in gdf_enf.columns:
    gdf_enf = gdf_enf.drop(columns=['grid_id'])

# 포인트가 어느 격자 폴리곤 안에 속하는지 조인
gdf_mapped = gpd.sjoin(gdf_enf, gdf_grids[['grid_id', 'geometry']], how='left', predicate='within')

# 4. 최종 저장
gdf_mapped = gdf_mapped.drop(columns=['geometry', 'index_right'], errors='ignore')
output_path = "enforcement_with_grid_id_fixed.csv"
gdf_mapped.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"🎉 공간 왜곡을 일으키던 좌표 수정 및 격자 재매핑이 완벽하게 끝났습니다!\n💾 저장 완료: {output_path}")

✅ 총 14건의 잘못된 좌표가 정상적으로 수정되었습니다.
🔄 수정된 좌표를 바탕으로 올바른 격자(Grid)를 다시 찾고 있습니다...
🎉 공간 왜곡을 일으키던 좌표 수정 및 격자 재매핑이 완벽하게 끝났습니다!
💾 저장 완료: enforcement_with_grid_id_fixed.csv
